# CFNet bi-temporal change detection — DIMER E2E fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/cfnet-change-detection-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/cfnet-change-detection-pipeline/blob/main/tutorials/cfnet_change_detection_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-wifibk%2FCFNet-ffcc4d?style=flat)](https://huggingface.co/wifibk/CFNet) [![Upstream](https://img.shields.io/badge/Upstream-wifiBlack%2FCFNet-181717?style=flat&logo=github&logoColor=white)](https://github.com/wifiBlack/CFNet) [![Paper](https://img.shields.io/badge/arXiv-2503.08505-b31b1b.svg)](https://arxiv.org/abs/2503.08505)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** binary building-change maps for pairs of co-registered 0.5 m RGB patches with CFNet's content-focuser network, held-out F1/IoU against an all-unchanged baseline, and bounded fine-tuning of the change decoder to labelled pairs

**This notebook is standalone.** It carries the repository's package (4 modules under `src/cfnet_change_detection_pipeline/`, at revision `71983b082c4c`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `c23279428d14186d67ce199b3db358038bf37585` (~16 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh runtime (GPU recommended, CPU works) installs the pinned dependencies (torch, torchvision, numpy, pillow, safetensors, huggingface-hub — the network is carried in this notebook, the EfficientNet-B5 stages come from the installed torchvision with no download), stages and digest-verifies the pinned CFNet LEVIR-CD checkpoint (15.8 MB) from the Hub, statically audits the pickle against an allow-list, converts it once into safetensors with a pinned digest, rebuilds the network from the carried module and loads it strictly, fetches the digest-pinned LEVIR-CD tarball (3.8 GB, no credential) and extracts exactly the 192 pinned before / after / label members, validates them and assigns the dataset's own splits (32 training, 8 validation, 24 test crops), detects change on the held-out pairs with the frozen model and scores them against the all-unchanged baseline, runs a bounded fine-tuning of the change decoder, scores the same pairs again, writes change maps for two held-out pairs beside their images and labels, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify prediction parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On a T4 the whole path takes under a minute of model time after the downloads; the 3.8 GB tarball is the slowest step.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own labelled pairs as a zip holding `pairs.csv` (columns `id`, `before`, `after`, `label`) beside co-registered RGB PNG / JPEG images of the same size (sides multiples of 32) and single-band label PNGs (0 = unchanged, 255 = changed); at least four pairs with some change. Your pairs are split by seed into training, validation and test sets and flow through the same contract — validation, frozen baseline, adaptation, held-out evaluation, change maps, artifact export and reload parity. The expected schema, the ceilings and the privacy guidance are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

CFNet (Wu et al., 2025) is a content-focuser network for bi-temporal remote-sensing change detection: a shared EfficientNet-B5 encoder over both dates, a content decoder per date trained to keep what is intrinsic to the scene rather than to its acquisition, a focuser that turns the cosine distance between the two dates' content maps into per-scale change weights, and a change decoder that fuses the dates with 3-D convolutions and decodes one change map. The checkpoint packaged here is the authors' LEVIR-CD model — trained on 0.5 m Google Earth patches of Texas cities taken 5–14 years apart with building-change labels (Chen and Shi, 2020) — which they report at F1 92.18 / IoU 85.49 on that dataset's test split.

Three things about this row are handled in the open. **The upstream asset is a pickle** — a torch state dict saved with `torch.save`. Section 3 downloads and digest-verifies it, statically lists every global the pickle would import (a state dict of tensors and nothing else), refuses anything outside that allow-list, unpickles it exactly once through torch's weights-only loader, and writes a safetensors file whose digest is pinned in the carried module; the network you run is rebuilt from `modeling.py`, carried in this notebook, with the EfficientNet-B5 stages taken from the installed torchvision package (`weights=None`, no download) and loads that file strictly. **The dataset ships as one 3.8 GB tarball under academic-only terms**: LEVIR-CD may be used for academic purposes only, commercial use is prohibited, and the imagery is subject to Google Earth's terms — so Section 4 pins the authors' mirror of the tarball by size and digest, streams through it once and copies out exactly the 192 pinned members (each pinned again by size and digest, no `extractall`, no paths taken from the archive), leaves the other 47,589 alone, and redistributes nothing. **The model was trained on this dataset**, so the bounded adaptation in Section 6 is a demonstration of the contract on the training split's own crops, selected by validation loss with the frozen model as epoch 0, and the test split — which the model never trained on — is the held-out set; the point of the contract is the same recipe applied to *your* labelled pairs.

**Learning objectives:** install the pinned runtime; inspect the carried network, pipeline, dataset and metrics modules; stage and digest-verify a pickled checkpoint, read its static audit and see it converted into safetensors; extract pinned members from a digest-verified tarball and validate real labelled bi-temporal pairs with an ignore class; read F1, IoU, precision and recall of the changed class against an all-unchanged baseline; run a bounded fine-tuning of the change decoder with the upstream loss, explicit hyperparameters and frozen BatchNorm statistics; compare the adapted and frozen models on the same held-out pairs; write change maps; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** multi-class or semantic change, change between more than two dates, images that are not co-registered, resolutions far from 0.5 m, tiling of scenes larger than a patch, the published benchmark scores, the CLCD and SYSU-CD checkpoints of the same repository, and any claim that a 64-crop sample stands in for an operational evaluation. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab, or a Jupyter kernel with Python 3.12); a GPU (T4 or better) makes the model time a few seconds, a CPU a few minutes — the network has 3.8 M parameters. About 4 GB of disk is needed for the tarball; the checkpoint and its conversion are 31 MB together.
- **Knowledge:** what a co-registered bi-temporal image pair is, what a binary change mask and an ignore class are, and how F1, IoU, precision and recall of a rare class are read against a majority baseline.
- **Executable serialization handled explicitly:** the pinned checkpoint is a pickle. It is digest-verified, statically audited against an allow-list (audit digest pinned) and unpickled **once** through torch's weights-only loader to produce the safetensors the network is actually loaded from. No Hub-hosted Python module is imported; the network is the carried `modeling.py` plus torchvision's EfficientNet-B5 stages built without weights.
- **Data contract:** a record is `{{id, before, after, label}}` — two (H, W, 3) uint8 RGB images (or PNG / JPEG paths) of the same size, sides in [64, 2048] and multiples of 32, and an (H, W) mask with 0 / 1 / −1 (or a PNG with 0 / 255). The pipeline reorders the channels to BGR and standardises each date with its own LEVIR-CD statistics, exactly as the upstream loader did. Validation is structural: nothing checks that the two images show the same place, that they are co-registered, that the resolution is about 0.5 m, or that the label belongs to the pair.
- **Privacy and terms:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — commercial imagery under licence is exactly that. The default path uploads nothing. LEVIR-CD, which the default path fetches, may be used **for academic purposes only**; commercial use is prohibited and the imagery is subject to Google Earth's terms of use.
- **External access (data):** besides the model snapshot, the default path fetches one pinned object — the 3.8 GB `LEVIR-CD-processed.tar.gz` of the Hugging Face dataset `wifibk/CFNet_Datasets` (the CFNet authors' mirror) at an immutable revision — over HTTPS, digest-verified before any member is read.
- **External access:** the Hugging Face Hub only, to fetch the pinned `wifibk/CFNet` snapshot (~16 MB in total) at revision `c23279428d14…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `torchvision`, `PIL` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'safetensors==0.8.0',
    'huggingface-hub==1.32.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'cfnet-change-detection-pipeline',
    'repository_revision': '71983b082c4cc65d18b1710565c52a39fb80cbbf',
    'embedded_module': 'src/cfnet_change_detection_pipeline/pipeline.py',
    'embedded_modules': ['src/cfnet_change_detection_pipeline/metrics.py', 'src/cfnet_change_detection_pipeline/modeling.py', 'src/cfnet_change_detection_pipeline/pipeline.py', 'src/cfnet_change_detection_pipeline/samples.py'],
    'module_sha256': '6f38fadd46cd992c5e68d1b9fc318b9557f0e434d5777679e2ac5739b1bbaadb',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, torchvision, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/cfnet_change_detection_pipeline/` @ `71983b082c4c`)

The next 4 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/4:** `src/cfnet_change_detection_pipeline/metrics.py`

In [ ]:
"""Pixel-level binary change-detection metrics for the tutorial and its constant baseline.

The upstream evaluation (`confuse_matrix.py`) pools one 2 × 2 confusion matrix over all scored pixels with
`pred > 0.5` and `target > 0.5` and reports recall, precision, F1 and IoU of the *change* class; this module
reproduces those four numbers on the same pooled matrix and adds the overall pixel accuracy, the change fraction of
the labels and of the predictions, and the confusion counts. All numbers are pooled over the labelled pixels of
the pairs scored together (the ignore index excluded); nothing here estimates dispersion.
"""

from __future__ import annotations

from collections.abc import Sequence
from typing import Any


def confusion_counts(predictions: Sequence[Any], labels: Sequence[Any], *, ignore_index: int) -> dict[str, int]:
    """TP / FP / FN / TN of the change class over every labelled pixel of the given pairs."""
    import numpy as np

    tp = fp = fn = tn = 0
    for pred, label in zip(predictions, labels, strict=True):
        pred = np.asarray(pred).reshape(-1)
        label = np.asarray(label).reshape(-1)
        if pred.shape != label.shape:
            raise ValueError(f"prediction shape {pred.shape} != label shape {label.shape}")
        valid = label != ignore_index
        p = pred[valid].astype(bool)
        t = label[valid].astype(bool)
        tp += int(np.sum(p & t))
        fp += int(np.sum(p & ~t))
        fn += int(np.sum(~p & t))
        tn += int(np.sum(~p & ~t))
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn}


def metrics_from_counts(counts: dict[str, int]) -> dict[str, Any]:
    tp, fp, fn, tn = counts["tp"], counts["fp"], counts["fn"], counts["tn"]
    total = tp + fp + fn + tn
    recall = tp / (tp + fn) if tp + fn else None
    precision = tp / (tp + fp) if tp + fp else None
    f1 = (
        2 * recall * precision / (recall + precision)
        if recall is not None and precision is not None and recall + precision
        else 0.0
    )
    iou = tp / (tp + fp + fn) if tp + fp + fn else None
    rnd = lambda v: None if v is None else round(float(v), 4)  # noqa: E731
    return {
        "pixels": total,
        "f1": rnd(f1),
        "iou": rnd(iou),
        "precision": rnd(precision),
        "recall": rnd(recall),
        "accuracy": rnd((tp + tn) / total) if total else None,
        "change_fraction_label": rnd((tp + fn) / total) if total else None,
        "change_fraction_predicted": rnd((tp + fp) / total) if total else None,
        "confusion": dict(counts),
    }


def change_metrics(predictions: Sequence[Any], labels: Sequence[Any], *, ignore_index: int = -1) -> dict[str, Any]:
    """F1, IoU, precision and recall of the change class (upstream's protocol), overall accuracy and the change
    fractions, pooled over the labelled pixels of the pairs."""
    return metrics_from_counts(confusion_counts(predictions, labels, ignore_index=ignore_index))


def unchanged_baseline(labels: Sequence[Any], *, ignore_index: int = -1) -> dict[str, Any]:
    """The constant predictor that calls every pixel unchanged — the baseline any change detector must beat —
    scored on the same pixels as the model: its accuracy is the unchanged fraction and its F1 and IoU are 0."""
    import numpy as np

    predictions = [np.zeros(np.asarray(label).shape, dtype=bool) for label in labels]
    report = change_metrics(predictions, labels, ignore_index=ignore_index)
    report["note"] = "predicts 'unchanged' everywhere; accuracy equals the unchanged fraction, F1 and IoU are 0"
    return report

**Module 2/4:** `src/cfnet_change_detection_pipeline/modeling.py` (carried verbatim; see the note above)

In [ ]:
"""CFNet — Content Focuser Network for bi-temporal remote-sensing change detection — in plain PyTorch.

Vendored from the authors' training code (`wifiBlack/CFNet`, `model/{CFNet,encoder,content_decoder,change_decoder,
focuser,utils}.py` at commit `54acadab23b9d9395ec6814386c2d4a1253eac5a`, Apache-2.0), rewritten as one module with
no training-time scaffolding (no argument parsing, no `autocast` inside `forward`, no feature-map dumps). The only
external component is the encoder backbone, the stem and first four stages of torchvision's EfficientNet-B5
(`torchvision.models.efficientnet_b5(weights=None).features[0:5]`), taken from the installed `torchvision` package
at a pinned version and never downloaded — the fine-tuned checkpoint carries its weights. Parameter and buffer
names reproduce the upstream state dict exactly (`encoder._backbone.*`, `content_decoder_1.*`,
`content_decoder_2.*`, `change_decoder.*`), which is how the conversion can load it with `strict=True`.

Architecture (Wu et al., 2025):

* `Encoder` — the shared EfficientNet-B5 stem + stages 1–4 applied to both dates; the outputs of stages 1–4
  (24 / 40 / 64 / 128 channels at 1/2, 1/4, 1/8 and 1/16 of the input) are kept per date.
* `ContentDecoder` (one per date) — a top-down path of three `Aggregation` blocks (transposed-convolution
  upsampling, 1 × 1 fusion, two residual blocks) producing four content maps from coarse to fine (128 → 64 → 40 →
  24 channels). Its CBAM attention modules are constructed and carry weights in the checkpoint but, as upstream,
  their outputs are never used (`forward` discards them), so this port keeps the modules and does not apply them.
* `Focuser` — per scale, the cosine distance between the two dates' content maps through `tanh` gives a
  change-focus map in [0, 1) that reweights the fused features.
* `ChangeDecoder` — per scale, a 3-D fusion convolution over the two dates, weighted by the focus map and
  aggregated top-down as in the content decoder; a final stride-2 transposed convolution returns a one-channel map
  at the input resolution through `tanh`. A fifth fusion block for the 3-channel input level exists in the checkpoint
  and, as upstream, is never called.

The network takes two standardised (B, 3, H, W) images — H and W multiples of 32 — and returns the change map
(B, H, W) in (−1, 1); upstream declares a pixel changed where the map exceeds 0.5.
"""

from __future__ import annotations

import torch
from torch import nn

CHANNELS: tuple[int, ...] = (3, 24, 40, 64, 128)  # input, then the four kept EfficientNet-B5 stages
BACKBONE_STAGES = 5  # features[0] (stem) + stages 1..4


def _efficientnet_b5_features() -> nn.ModuleList:
    """The stem and first four stages of torchvision's EfficientNet-B5, randomly initialised (no download)."""
    from torchvision.models import efficientnet_b5

    features = efficientnet_b5(weights=None).features
    return nn.ModuleList(list(features.children())[:BACKBONE_STAGES])


class Encoder(nn.Module):
    """Shared backbone applied to both dates; returns the four stage outputs per date (fine to coarse)."""

    def __init__(self) -> None:
        super().__init__()
        self._backbone = _efficientnet_b5_features()

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> tuple[list[torch.Tensor], list[torch.Tensor]]:
        y1: list[torch.Tensor] = []
        y2: list[torch.Tensor] = []
        for index, layer in enumerate(self._backbone):
            x1 = layer(x1)
            x2 = layer(x2)
            if index != 0:
                y1.append(x1)
                y2.append(x2)
        return y1, y2


class CBA1x1(nn.Module):
    """1 × 1 convolution → BatchNorm → ReLU (upstream `CBA1x1`)."""

    def __init__(self, in_channel: int, out_channel: int) -> None:
        super().__init__()
        self.block = nn.Sequential(nn.Conv2d(in_channel, out_channel, kernel_size=1), nn.BatchNorm2d(out_channel), nn.ReLU())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class BasicBlock(nn.Module):
    """ResNet-style residual block (upstream `BasicBlock`, no downsampling)."""

    def __init__(self, in_channel: int, out_channel: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_channel, out_channel, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channel)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(out_channel, out_channel, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channel)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + x)


class Aggregation(nn.Module):
    """Upsample the coarse map by a stride-2 transposed convolution, concatenate the finer map, fuse 1 × 1, two
    residual blocks (upstream `Aggregation`)."""

    def __init__(self, in_channel: int, out_channel: int) -> None:
        super().__init__()
        self.upsample = nn.ConvTranspose2d(
            in_channel, in_channel, kernel_size=3, stride=2, padding=1, output_padding=1, bias=False
        )
        self.conv1 = CBA1x1(in_channel + out_channel, out_channel)
        self.residual1 = BasicBlock(out_channel, out_channel)
        self.residual2 = BasicBlock(out_channel, out_channel)

    def forward(self, coarse: torch.Tensor, fine: torch.Tensor) -> torch.Tensor:
        y = self.conv1(torch.cat([self.upsample(coarse), fine], dim=1))
        return self.residual2(self.residual1(y))


class ChannelAttention(nn.Module):
    def __init__(self, in_planes: int, ratio: int = 16) -> None:
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1 = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv1(torch.cat([avg_out, max_out], dim=1)))


class CBAM(nn.Module):
    """Convolutional block attention (upstream `CBAM`); present in the checkpoint, unused by the forward pass."""

    def __init__(self, in_planes: int, ratio: int = 16, kernel_size: int = 7) -> None:
        super().__init__()
        self.channel_attention = ChannelAttention(in_planes, ratio)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_out = self.channel_attention(x) * x
        return self.spatial_attention(x_out) * x_out


class ContentDecoder(nn.Module):
    """Top-down aggregation of one date's four stage outputs into four content maps, coarse to fine."""

    def __init__(self, channel_list: tuple[int, ...] = CHANNELS) -> None:
        super().__init__()
        coarse_to_fine = tuple(reversed(channel_list))  # 128, 64, 40, 24, 3
        targets = tuple(reversed(channel_list[1:-1]))  # 64, 40, 24
        self.aggregations = nn.ModuleList([Aggregation(i, o) for i, o in zip(coarse_to_fine, targets, strict=False)])
        self.cbam = nn.ModuleList([CBAM(c) for c in reversed(channel_list[1:])])  # weights present, never applied

    def forward(self, features: list[torch.Tensor]) -> list[torch.Tensor]:
        maps = [features[-1]]
        for index, fine in enumerate(reversed(features[:-1])):
            maps.append(self.aggregations[index](maps[index], fine))
        # Upstream applies `self.cbam[idx]` to each map here and discards the result; the maps are returned as they are.
        return maps


class Focuser(nn.Module):
    """Per-scale change-focus maps: tanh of the cosine distance between the two dates' content maps."""

    def __init__(self) -> None:
        super().__init__()
        self.cos_sim = nn.CosineSimilarity(dim=1)

    def forward(self, maps_1: list[torch.Tensor], maps_2: list[torch.Tensor]) -> list[torch.Tensor]:
        return [torch.tanh(1.0 - self.cos_sim(a, b)) for a, b in zip(maps_1, maps_2, strict=True)]


class FuseConv3d(nn.Module):
    """Fuse the two dates' maps at one scale with a (2, 3, 3) 3-D convolution → BatchNorm3d → ReLU."""

    def __init__(self, channels: int) -> None:
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(channels, channels, kernel_size=(2, 3, 3), stride=(2, 1, 1), padding=(0, 1, 1)),
            nn.BatchNorm3d(channels),
            nn.ReLU(),
        )

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x = torch.cat([x1.unsqueeze(2), x2.unsqueeze(2)], dim=2)
        return self.conv(x).squeeze(2)


class ChangeDecoder(nn.Module):
    """Fuse both dates per scale, weight by the focus maps, aggregate top-down, and decode one change map."""

    def __init__(self, channel_list: tuple[int, ...] = CHANNELS) -> None:
        super().__init__()
        coarse_to_fine = tuple(reversed(channel_list))  # 128, 64, 40, 24, 3 — the last (3) is never called
        targets = tuple(reversed(channel_list[1:-1]))
        self.fuseconv3ds = nn.ModuleList([FuseConv3d(c) for c in coarse_to_fine])
        self.aggregations = nn.ModuleList([Aggregation(i, o) for i, o in zip(coarse_to_fine, targets, strict=False)])
        self.upconv = nn.Sequential(
            nn.ConvTranspose2d(channel_list[1], 1, kernel_size=3, stride=2, padding=1, output_padding=1, bias=False)
        )

    def forward(self, maps_1: list[torch.Tensor], maps_2: list[torch.Tensor], focuses: list[torch.Tensor]) -> torch.Tensor:
        fused: list[torch.Tensor] = []
        for index, (a, b, focus) in enumerate(zip(maps_1, maps_2, focuses, strict=True)):
            y = self.fuseconv3ds[index](a, b) * focus.unsqueeze(1)
            if index > 0:
                y = self.aggregations[index - 1](fused[index - 1], y)
            fused.append(y)
        return torch.tanh(self.upconv(fused[-1]).squeeze(1))


class CFNet(nn.Module):
    """Two standardised (B, 3, H, W) dates → (B, H, W) change map in (−1, 1)."""

    def __init__(self) -> None:
        super().__init__()
        self.encoder = Encoder()
        self.content_decoder_1 = ContentDecoder()
        self.content_decoder_2 = ContentDecoder()
        self.change_decoder = ChangeDecoder()
        self.focuser = Focuser()

    def content(self, x1: torch.Tensor, x2: torch.Tensor) -> tuple[list[torch.Tensor], list[torch.Tensor]]:
        """The two dates' content maps (four scales each, coarse to fine)."""
        y1, y2 = self.encoder(x1, x2)
        return self.content_decoder_1(y1), self.content_decoder_2(y2)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        if tuple(x1.shape) != tuple(x2.shape):
            raise ValueError(f"the two dates must have the same shape, got {tuple(x1.shape)} and {tuple(x2.shape)}")
        if x1.shape[-1] % 32 or x1.shape[-2] % 32:
            raise ValueError(f"height and width must be multiples of 32, got {tuple(x1.shape[-2:])}")
        maps_1, maps_2 = self.content(x1, x2)
        focuses = self.focuser(maps_1, maps_2)
        return self.change_decoder(maps_1, maps_2, focuses)

    def forward_with_content(
        self, x1: torch.Tensor, x2: torch.Tensor
    ) -> tuple[torch.Tensor, list[torch.Tensor], list[torch.Tensor], list[torch.Tensor]]:
        """Change map plus the content maps and focus maps — what the upstream loss consumes."""
        maps_1, maps_2 = self.content(x1, x2)
        focuses = self.focuser(maps_1, maps_2)
        return self.change_decoder(maps_1, maps_2, focuses), maps_1, maps_2, focuses


def check_shapes(module: nn.Module) -> dict[str, int]:
    """Sanity numbers used by the tests and the conversion: tensors, elements and parameters."""
    state = module.state_dict()
    return {
        "tensors": len(state),
        "elements": sum(int(v.numel()) for v in state.values()),
        "parameters": sum(p.numel() for p in module.parameters()),
    }

**Module 3/4:** `src/cfnet_change_detection_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""CFNet bi-temporal change detection (`wifibk/CFNet`, the LEVIR-CD checkpoint) DIMER pipeline: verified snapshot,
one-time conversion of the pickled state dict into safetensors, binary building-change maps for pairs of
co-registered RGB patches, held-out evaluation against an all-unchanged baseline, and bounded fine-tuning of the
change decoder to a user's labelled pairs with a portable adapter.

CFNet (Wu et al., 2025) is a content-focuser network: a shared EfficientNet-B5 encoder over both dates, one
content decoder per date, a focuser that turns the cosine distance between the dates' content maps into per-scale
change weights, and a change decoder that fuses the two dates with 3-D convolutions and decodes one change map.
The checkpoint packaged here, `levir-cd.pth`, is the authors' model trained on LEVIR-CD — 0.5 m Google Earth
patches of Texas cities with building-change labels (Chen and Shi, 2020) — which they report at F1 92.18 / IoU
85.49 on that dataset's test split.

The upstream asset is a torch zip archive whose pickle references only `collections.OrderedDict`,
`torch._utils._rebuild_tensor_v2` and two storage classes (verified statically by `audit_pickle`). Under the fleet
asset specification (§11) that is executable serialization, so this package converts it once —
`torch.load(weights_only=True)`, a strict load into the vendored architecture — into safetensors with a pinned
digest, and serves only the converted file. The architecture is `modeling.py`, plain PyTorch plus the stem and four
stages of torchvision's EfficientNet-B5 built with `weights=None`; nothing is fetched from the Hub at load time
except the manifest-listed files.

Everything model-related is imported lazily so that snapshot verification, the pickle audit and input validation
run (and can refuse) before `torch` or `torchvision` are imported (fleet RTM-001). `numpy` and `PIL` are used for
images and are imported freely.
"""

from __future__ import annotations

import hashlib
import io
import json
import math
import pickletools
import time
import zipfile
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

MODEL_ID = "wifibk/CFNet"
MODEL_REVISION = "c23279428d14186d67ce199b3db358038bf37585"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "cfnet-levir-cd"
ARTIFACT_FORMAT = "org.valcorza.cfnet-change-detection.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Immutable upstream source asset (a torch pickle of the state dict; see docs/WEIGHTS.md).
SOURCE_CKPT_NAME = "levir-cd.pth"
SOURCE_CKPT_BYTES = 15_805_666
SOURCE_CKPT_SHA256 = "22ab286b2138ab1082e04290b27cbff5abd3802375a6b2f01d4ba78ca8b0c1aa"
# Code-free serving file produced deterministically by `convert_model` (asset spec §11.2).
CONVERTED_WEIGHTS_NAME = "cfnet-levir-cd.safetensors"
CONVERTED_SHA256 = "b348186e5003a7447ab5eb5874204c50e63ad71aa61aca3ff0a80746116963d9"
CONVERTED_BYTES = 15_598_980
# Static-audit digest of the source pickle (sorted global names), see `audit_pickle`.
PICKLE_AUDIT_SHA256 = "5b9f0ba08490293d6c17b9cef219991e1a6edda31609429679f8dca1af5a7b10"
CKPT_ALLOWED_GLOBALS = frozenset(
    {"collections.OrderedDict", "torch._utils._rebuild_tensor_v2", "torch.FloatStorage", "torch.LongStorage"}
)

# Architecture and data-contract facts (upstream `model/` and `dataset/dataset.py` at the vendored commit).
UPSTREAM_CODE_COMMIT = "54acadab23b9d9395ec6814386c2d4a1253eac5a"
STATE_TENSORS = 776
STATE_NUMEL = 3_877_781
PARAMETER_COUNT = 3_838_563  # nn.Parameters; the rest are BatchNorm buffers
ENCODER_TENSORS = 428  # torchvision efficientnet_b5.features[0:5]
NUM_CLASSES = 2
CLASS_NAMES: tuple[str, ...] = ("unchanged", "changed")
IGNORE_INDEX = -1
CHANGE_THRESHOLD = 0.5  # upstream: a pixel is changed where the tanh change map exceeds 0.5
# Upstream standardises each date separately, after `cv2.imread` (BGR channel order) and division by 255, with
# the LEVIR-CD statistics of `dataset/dataset.py`; the tuples below are in BGR order, as the code applied them.
MEANS_BEFORE: tuple[float, ...] = (0.45028868, 0.44673658, 0.38134101)
STDS_BEFORE: tuple[float, ...] = (0.17450373, 0.16485656, 0.15314709)
MEANS_AFTER: tuple[float, ...] = (0.34578751, 0.33841163, 0.28902323)
STDS_AFTER: tuple[float, ...] = (0.1292925, 0.12592704, 0.11870409)
MIN_SIDE = 64
MAX_SIDE = 2048
SIDE_MULTIPLE = 32  # four stride-2 stages plus the stem
SAMPLE_SIZE = 256  # the LEVIR-CD crops of the tutorial
MIN_RECORDS = 4
MAX_RECORDS = 2_000
ADAPTATION_MODES = ("change_decoder", "decoders")  # the only scopes an adapter may declare
TRAINABLE_PREFIXES: dict[str, tuple[str, ...]] = {
    "change_decoder": ("change_decoder.",),
    "decoders": ("change_decoder.", "content_decoder_1.", "content_decoder_2."),
}
CONTENT_LOSS_WEIGHT = 0.1  # upstream `beta`; the change loss weight `alpha` is 1


# --------------------------------------------------------------------------------------------------
# manifest, staging, static pickle audit and conversion
# --------------------------------------------------------------------------------------------------


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"no snapshot manifest at {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    listed = {entry["path"] for entry in manifest["files"]}
    if SOURCE_CKPT_NAME not in listed:
        raise ValueError(f"manifest does not list {SOURCE_CKPT_NAME}; refusing to proceed")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256_file(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
        if entry["path"] == SOURCE_CKPT_NAME and (size, digest) != (SOURCE_CKPT_BYTES, SOURCE_CKPT_SHA256):
            raise ValueError(f"{entry['path']}: manifest digest disagrees with the package constant")
    return manifest


def verify_converted(path: str | Path | None = None) -> dict[str, Any]:
    """Check the converted serving file (safetensors) against the pinned digest."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    file_path = root / CONVERTED_WEIGHTS_NAME
    if not file_path.is_file():
        raise FileNotFoundError(f"converted file missing: {file_path}")
    size = file_path.stat().st_size
    if size != CONVERTED_BYTES:
        raise ValueError(f"{CONVERTED_WEIGHTS_NAME}: size {size} != pinned {CONVERTED_BYTES}")
    digest = _sha256_file(file_path)
    if digest != CONVERTED_SHA256:
        raise ValueError(f"{CONVERTED_WEIGHTS_NAME}: sha256 {digest} != pinned {CONVERTED_SHA256}")
    return {"files": [{"path": CONVERTED_WEIGHTS_NAME, "bytes": size, "sha256": digest}]}


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the snapshot against its DIMER manifest (size + SHA-256 of every listed Hub file) and, when the
    converted serving file is present, that against the pinned digest."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _verify_manifest(root, MODEL_ID, MODEL_REVISION)
    converted = (root / CONVERTED_WEIGHTS_NAME).is_file()
    if converted:
        verify_converted(root)
    return {**manifest, "converted": converted}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest entries that are absent locally (a fresh clone commits the manifest and git-ignores the
    checkpoint and the safetensors it converts to)."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _pickle_globals(data: bytes) -> dict[str, int]:
    """Every global a pickle stream would import, collected with `pickletools.genops` (no execution)."""
    found: dict[str, int] = {}
    stack: list[Any] = []
    for op, arg, _pos in pickletools.genops(io.BytesIO(data)):
        if op.name == "GLOBAL":  # pickletools renders the (module, name) pair space-separated
            key = arg.replace("\n", " ").replace(" ", ".", 1)
            found[key] = found.get(key, 0) + 1
        elif op.name == "STACK_GLOBAL":
            key = f"{stack[-2]}.{stack[-1]}"
            found[key] = found.get(key, 0) + 1
        if op.name in ("SHORT_BINUNICODE", "BINUNICODE", "UNICODE", "SHORT_BINSTRING", "BINSTRING"):
            stack.append(arg)
        elif op.name in ("MEMOIZE", "BINPUT", "LONG_BINPUT", "PUT"):
            pass
        else:
            stack.append(None)
    return found


def audit_pickle(path: str | Path, *, allowed: frozenset[str] = CKPT_ALLOWED_GLOBALS) -> dict[str, Any]:
    """Statically list the globals a pickle (plain, or inside a torch zip archive) would import and refuse any
    outside `allowed`. Executes nothing. Returns the sorted globals and their digest."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"file not found: {file_path}")
    data = file_path.read_bytes()
    found: dict[str, int] = {}
    nested = 0
    if data[:4] == b"PK\x03\x04":
        archive = zipfile.ZipFile(io.BytesIO(data))
        for name in archive.namelist():
            if name.endswith(".pkl"):
                nested += 1
                for key, count in _pickle_globals(archive.read(name)).items():
                    found[key] = found.get(key, 0) + count
    else:
        found = _pickle_globals(data)
    violations = sorted(name for name in found if name not in allowed)
    summary = {
        "file": file_path.name,
        "torch_archive": data[:4] == b"PK\x03\x04",
        "pickles": nested if nested else 1,
        "globals": sorted(found),
        "violations": violations,
        "audit_sha256": hashlib.sha256("\n".join(sorted(found)).encode("utf-8")).hexdigest(),
    }
    if violations:
        raise ValueError(f"{file_path.name}: pickle audit failed, globals outside the allow-list: {violations}")
    return summary


def _check_pinned_source(root: Path) -> dict[str, Any]:
    source = root / SOURCE_CKPT_NAME
    if not source.is_file():
        raise FileNotFoundError(f"source file not found: {source}")
    size = source.stat().st_size
    if size != SOURCE_CKPT_BYTES:
        raise ValueError(f"{SOURCE_CKPT_NAME}: size {size} != pinned {SOURCE_CKPT_BYTES}")
    digest = _sha256_file(source)
    if digest != SOURCE_CKPT_SHA256:
        raise ValueError(f"{SOURCE_CKPT_NAME}: sha256 {digest} != pinned {SOURCE_CKPT_SHA256}")
    audit = audit_pickle(source)
    if audit["audit_sha256"] != PICKLE_AUDIT_SHA256:
        raise ValueError(f"{SOURCE_CKPT_NAME}: pickle audit digest {audit['audit_sha256']} != pinned {PICKLE_AUDIT_SHA256}")
    return {"path": SOURCE_CKPT_NAME, "bytes": size, "sha256": digest, "audit": audit}


def build_model() -> Any:
    """Instantiate the architecture from the vendored module (the EfficientNet-B5 stages from the installed
    torchvision package with `weights=None`; no download)."""
    pass  # standalone rewrite (build_notebook.py): `from .modeling import CFNet` removed — names are kernel globals defined by the carried modules

    return CFNet()


def convert_model(path: str | Path | None = None) -> dict[str, Any]:
    """Convert the pinned checkpoint into safetensors, deterministically, after size, digest and static-audit
    checks: torch's weights-only unpickler, a strict load into the vendored architecture, and the model's own
    state dict saved."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    source = _check_pinned_source(root)
    import torch
    from safetensors.torch import save_file

    started = time.perf_counter()
    checkpoint = root / SOURCE_CKPT_NAME
    state = torch.load(checkpoint, map_location="cpu", weights_only=True)
    if not isinstance(state, dict) or any(not isinstance(v, torch.Tensor) for v in state.values()):
        raise ValueError(f"{SOURCE_CKPT_NAME} did not unpickle to a state dict of tensors")
    if len(state) != STATE_TENSORS:
        raise ValueError(f"{SOURCE_CKPT_NAME}: state dict has {len(state)} tensors, expected {STATE_TENSORS}")
    model = build_model()
    model.load_state_dict(state, strict=True)
    canonical = {k: v.contiguous() for k, v in model.state_dict().items()}
    n_elements = sum(v.numel() for v in canonical.values())
    if len(canonical) != STATE_TENSORS or n_elements != STATE_NUMEL:
        raise ValueError(
            f"converted state dict has {len(canonical)} tensors / {n_elements} elements; expected {STATE_TENSORS} / {STATE_NUMEL}"
        )
    save_file(canonical, str(root / CONVERTED_WEIGHTS_NAME), metadata={"format": "pt"})
    report = verify_converted(root)
    return {
        "source": {k: v for k, v in source.items() if k != "audit"},
        "audit": source["audit"],
        "checkpoint": {
            "entries": "a plain state dict (no optimizer, no metadata)",
            "state_dict_tensors": len(state),
            "encoder_tensors": sum(1 for k in state if k.startswith("encoder.")),
        },
        "converted": report["files"],
        "seconds": round(time.perf_counter() - started, 2),
    }


# --------------------------------------------------------------------------------------------------
# image pairs, labels and validation (no model import)
# --------------------------------------------------------------------------------------------------

INPUT_SCHEMA: dict[str, Any] = {
    "record": (
        "{id, before, after, label?}: before / after = (H, W, 3) uint8 RGB images of the same co-registered footprint "
        "at two dates (or PNG / JPEG paths); label = (H, W) int mask with 0 = unchanged, 1 = changed, -1 = no data "
        "(or a PNG path with 0 / 255), optional"
    ),
    "image_size": (
        f"H and W in [{MIN_SIDE}, {MAX_SIDE}], both multiples of {SIDE_MULTIPLE}; the tutorial uses {SAMPLE_SIZE} × {SAMPLE_SIZE}"
    ),
    "value_units": (
        "8-bit RGB (the pipeline reorders to BGR and standardises each date with the LEVIR-CD statistics, as upstream)"
    ),
    "classes": {str(i): name for i, name in enumerate(CLASS_NAMES)},
    "ignore_index": IGNORE_INDEX,
    "decision_rule": f"changed where the tanh change map exceeds {CHANGE_THRESHOLD} (upstream)",
    "records": [MIN_RECORDS, MAX_RECORDS],
    "validation": (
        "record shape, dtype range, equal sizes of the two dates, side multiples and label values only. Nothing checks "
        "that the two images show the same place, that they are co-registered, that the resolution is about 0.5 m, "
        "or that the label was drawn for this pair -- any two same-sized RGB images are compared without complaint"
    ),
}


def read_image(path: str | Path) -> Any:
    """Load an RGB image from a PNG / JPEG as uint8 (H, W, 3); greyscale and RGBA are converted."""
    import numpy as np
    from PIL import Image

    with Image.open(path) as image:
        return np.ascontiguousarray(np.asarray(image.convert("RGB"), dtype=np.uint8))


def read_mask(path: str | Path) -> Any:
    """Load a label raster (0 = unchanged, 255 = changed; upstream thresholds at 127) as int64 (H, W) with 0 / 1."""
    import numpy as np
    from PIL import Image

    with Image.open(path) as image:
        array = np.asarray(image.convert("L"))
    return np.ascontiguousarray((array > 127).astype(np.int64))


def _check_image(image: Any, label_name: str, what: str) -> Any:
    import numpy as np

    if isinstance(image, str | Path):
        if not Path(image).is_file():
            raise ValueError(f"{label_name}: {what} file not found: {image}")
        image = read_image(image)
    try:
        array = np.asarray(image)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{label_name}: {what} must be an image array") from exc
    if array.ndim != 3 or array.shape[-1] != 3:
        raise ValueError(f"{label_name}: {what} must have shape (H, W, 3), got {array.shape}")
    if array.dtype != np.uint8:
        if not np.issubdtype(array.dtype, np.number) or float(array.min()) < 0 or float(array.max()) > 255:
            raise ValueError(f"{label_name}: {what} must be uint8 RGB or numeric in [0, 255]")
        array = np.rint(array).astype(np.uint8)
    height, width = array.shape[:2]
    if not (MIN_SIDE <= height <= MAX_SIDE and MIN_SIDE <= width <= MAX_SIDE):
        raise ValueError(f"{label_name}: {what} sides must be in [{MIN_SIDE}, {MAX_SIDE}], got {(height, width)}")
    if height % SIDE_MULTIPLE or width % SIDE_MULTIPLE:
        raise ValueError(f"{label_name}: {what} sides must be multiples of {SIDE_MULTIPLE}, got {(height, width)}")
    return np.ascontiguousarray(array)


def _check_record(record: Any, index: int) -> dict[str, Any]:
    import numpy as np

    label_name = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label_name} must be a mapping with id/before/after[/label]")
    for key in ("id", "before", "after"):
        if key not in record:
            raise ValueError(f"{label_name} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not rid or len(rid) > 128:
        raise ValueError(f"{label_name}: id must be a non-empty string of at most 128 characters")
    before = _check_image(record["before"], label_name, "before")
    after = _check_image(record["after"], label_name, "after")
    if before.shape != after.shape:
        raise ValueError(f"{label_name}: before and after must have the same shape, got {before.shape} and {after.shape}")
    item: dict[str, Any] = {"id": rid, "before": before, "after": after}
    label = record.get("label")
    if label is not None:
        if isinstance(label, str | Path):
            if not Path(label).is_file():
                raise ValueError(f"{label_name}: label file not found: {label}")
            label = read_mask(label)
        try:
            mask = np.asarray(label)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{label_name}: label must be an integer array") from exc
        if mask.shape != before.shape[:2]:
            raise ValueError(f"{label_name}: label must have shape {before.shape[:2]}, got {mask.shape}")
        if not np.issubdtype(mask.dtype, np.integer) and not np.all(mask == np.round(mask)):
            raise ValueError(f"{label_name}: label values must be integers")
        allowed = {0, 1, IGNORE_INDEX}
        found = set(np.unique(mask).astype(int).tolist())
        if not found <= allowed:
            raise ValueError(f"{label_name}: label values {sorted(found - allowed)} outside {sorted(allowed)}")
        item["label"] = np.ascontiguousarray(mask.astype(np.int64))
    for key in ("split", "region", "source", "source_id"):
        if key in record:
            item[key] = record[key]
    return item


def check_record(record: Mapping[str, Any]) -> dict[str, Any]:
    """Validate one record and return its normalised copy (uint8 RGB dates, int64 label)."""
    return _check_record(record, 0)


def pair_digest(record: Mapping[str, Any]) -> str:
    checked = _check_record(record, 0)
    digest = hashlib.sha256(checked["before"].tobytes())
    digest.update(checked["after"].tobytes())
    if "label" in checked:
        digest.update(checked["label"].tobytes())
    return digest.hexdigest()


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], pair_digest(r)] for r in records]
    return hashlib.sha256(json.dumps(payload, separators=(",", ":")).encode("utf-8")).hexdigest()


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    require_labels: bool = True,
) -> dict[str, Any]:
    """Structural validation of a pair dataset; raises ValueError before any model import."""

    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, before, after, label} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    changed = unchanged = ignored = 0
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        if require_labels and "label" not in item:
            raise ValueError(f"records[{index}] has no label; every record of a labelled dataset needs one")
        if "label" in item:
            changed += int((item["label"] == 1).sum())
            unchanged += int((item["label"] == 0).sum())
            ignored += int((item["label"] == IGNORE_INDEX).sum())
        checked.append(item)
    labelled = sum("label" in r for r in checked)
    if require_labels and labelled and changed == 0:
        raise ValueError("no changed pixel in the dataset; nothing to learn or evaluate")
    total = changed + unchanged
    sizes = sorted({tuple(r["before"].shape[:2]) for r in checked})
    return {
        "records": checked,
        "n_records": len(checked),
        "n_labelled": labelled,
        "sizes": [list(s) for s in sizes],
        "change_fraction": round(changed / total, 4) if total else None,
        "ignored_pixels": ignored,
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def validate_inputs(record: Mapping[str, Any]) -> dict[str, Any]:
    """Validate one record; returns its id, size, and label change fraction."""
    item = _check_record(record, 0)
    report = {"id": item["id"], "shape": tuple(item["before"].shape), "has_label": "label" in item}
    if "label" in item:
        label = item["label"]
        valid = int((label != IGNORE_INDEX).sum())
        report["change_fraction"] = round(float((label == 1).sum()) / max(valid, 1), 4)
        report["ignored_pixels"] = int((label == IGNORE_INDEX).sum())
    return report


# --------------------------------------------------------------------------------------------------
# pipeline
# --------------------------------------------------------------------------------------------------


def _normalise(images: Any, *, which: str) -> Any:
    """(B, H, W, 3) uint8 RGB -> (B, 3, H, W) float32 standardised as the upstream loader did: `cv2.imread`
    delivers BGR, so the channels are reversed, divided by 255 and standardised per date with the LEVIR-CD
    statistics (`which` = 'before' or 'after')."""
    import numpy as np

    mean, std = (MEANS_BEFORE, STDS_BEFORE) if which == "before" else (MEANS_AFTER, STDS_AFTER)
    batch = np.asarray(images, dtype=np.float32)[..., ::-1] / 255.0  # RGB -> BGR
    batch = np.transpose(batch, (0, 3, 1, 2))
    mean_a = np.asarray(mean, dtype=np.float32)[None, :, None, None]
    std_a = np.asarray(std, dtype=np.float32)[None, :, None, None]
    return np.ascontiguousarray((batch - mean_a) / std_a)


@dataclass
class CFNetChangePipeline:
    """Building-change maps and bounded change-decoder fine-tuning on top of the verified CFNet LEVIR-CD model."""

    model: Any
    device: str
    weights_dir: Path
    source: str
    adapter: dict[str, Any] | None = None

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        require_source: bool = True,
        report: Callable[[dict[str, Any]], None] | None = None,
    ) -> CFNetChangePipeline:
        """Verify, convert if needed, rebuild from the vendored module and strictly load. With
        `require_source=False` the checkpoint may be absent (the DIMER-hosted case) as long as the converted file
        verifies. `report` receives the audit and conversion records when a conversion happens."""
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if require_source:
            stage_missing_files(root, allow_download=allow_download)
            snapshot = verify_snapshot(root)
            if not snapshot["converted"]:
                conversion = convert_model(root)
                if report is not None:
                    report({"conversion": conversion})
                snapshot = verify_snapshot(root)
            elif report is not None:
                report({"conversion": "converted file already present and digest-verified"})
            source = "converted from the manifest-verified source checkpoint"
        else:
            verify_converted(root)
            source = "converted file, pinned digest (source checkpoint not required)"
        import torch
        from safetensors.torch import load_file

        chosen = device or ("cuda" if torch.cuda.is_available() else "cpu")
        if chosen.startswith("cuda") and not torch.cuda.is_available():
            raise ValueError("device='cuda' requested but CUDA is not available")
        model = build_model()
        state = load_file(str(root / CONVERTED_WEIGHTS_NAME))
        model.load_state_dict(state, strict=True)
        n_params = sum(p.numel() for p in model.parameters())
        if n_params != PARAMETER_COUNT:
            raise ValueError(f"rebuilt model has {n_params} parameters, expected {PARAMETER_COUNT}")
        model.to(torch.device(chosen)).eval()
        for param in model.parameters():
            param.requires_grad_(False)
        return cls(model=model, device=chosen, weights_dir=root, source=source)

    # ---- forward ---------------------------------------------------------------------------------------

    def _inputs(self, before: Any, after: Any) -> tuple[Any, Any]:
        import torch

        x1 = torch.from_numpy(_normalise(before, which="before")).to(self.device)
        x2 = torch.from_numpy(_normalise(after, which="after")).to(self.device)
        return x1, x2

    def _change_map(self, before: Any, after: Any, *, grad: bool = False, with_content: bool = False) -> Any:
        """(B, H, W, 3) uint8 pairs -> (B, H, W) float32 change map in (-1, 1) [and the content / focus maps]."""
        import torch

        x1, x2 = self._inputs(before, after)
        use_amp = self.device.startswith("cuda")
        context = torch.enable_grad() if grad else torch.inference_mode()
        with context, torch.autocast(device_type=self.device.split(":")[0], dtype=torch.float16, enabled=use_amp):
            if with_content:
                change, maps_1, maps_2, focuses = self.model.forward_with_content(x1, x2)
                return change.float(), maps_1, maps_2, focuses
            return self.model(x1, x2).float()

    # ---- inference -------------------------------------------------------------------------------------

    def predict(self, records: Sequence[Mapping[str, Any]], *, batch_size: int = 4) -> dict[str, Any]:
        """Detect change: per record the binary mask (H, W) uint8 (1 = changed, the upstream threshold on the
        change map), the change map (H, W) float32 in (-1, 1) — the network's output, not a calibrated
        probability — and the changed fraction."""
        import numpy as np

        checked = validate_dataset(records, min_records=1, require_labels=False)["records"]
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 32:
            raise ValueError("batch_size must be an int in 1..32")
        started = time.perf_counter()
        predictions = []
        groups: dict[tuple[int, ...], list[dict[str, Any]]] = {}
        for record in checked:  # batches hold one size at a time
            groups.setdefault(tuple(record["before"].shape), []).append(record)
        by_id: dict[str, dict[str, Any]] = {}
        for group in groups.values():
            for start in range(0, len(group), batch_size):
                batch = group[start : start + batch_size]
                maps = (
                    self._change_map(np.stack([r["before"] for r in batch]), np.stack([r["after"] for r in batch])).cpu().numpy()
                )
                for record, change_map in zip(batch, maps, strict=True):
                    mask = (change_map > CHANGE_THRESHOLD).astype(np.uint8)
                    by_id[record["id"]] = {
                        "id": record["id"],
                        "mask": mask,
                        "change_map": change_map.astype(np.float32),
                        "changed_fraction": round(float(mask.mean()), 4),
                    }
        predictions = [by_id[r["id"]] for r in checked]
        return {
            "model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": MODEL_KEY, "adapted": self.adapter is not None},
            "classes": list(CLASS_NAMES),
            "decision_rule": INPUT_SCHEMA["decision_rule"],
            "predictions": predictions,
            "seconds": round(time.perf_counter() - started, 3),
        }

    def evaluate(self, records: Sequence[Mapping[str, Any]], *, batch_size: int = 4) -> dict[str, Any]:
        """Pixel-level change metrics on labelled pairs (ignore index excluded): F1, IoU, precision and recall of
        the changed class as upstream scores them, plus accuracy and the change fractions, with the all-unchanged
        baseline scored on the same pixels."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import change_metrics, unchanged_baseline` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1)["records"]
        started = time.perf_counter()
        result = self.predict(checked, batch_size=batch_size)
        masks = [p["mask"] for p in result["predictions"]]
        labels = [r["label"] for r in checked]
        return {
            "n_records": len(checked),
            "metric": (
                "pixel F1 / IoU of the changed class over the labelled pixels of the held-out pairs (ignore index excluded)"
            ),
            "model": change_metrics(masks, labels, ignore_index=IGNORE_INDEX),
            "baseline_unchanged": unchanged_baseline(labels, ignore_index=IGNORE_INDEX),
            "adapted": self.adapter is not None,
            "seconds": round(time.perf_counter() - started, 3),
        }

    # ---- adaptation ------------------------------------------------------------------------------------

    def _trainable(self, mode: str) -> list[str]:
        if mode not in ADAPTATION_MODES:
            raise ValueError(f"trainable must be one of {ADAPTATION_MODES}")
        prefixes = TRAINABLE_PREFIXES[mode]
        return sorted(name for name, _param in self.model.named_parameters() if name.startswith(prefixes))

    @staticmethod
    def _content_loss(x1: Any, x2: Any, generator: Any, *, mode: str) -> Any:
        """Upstream `ContentLoss`, vectorised: for `n = W` random pixel pairs, the cosine similarity between the
        two pixels' feature vectors is computed within each date; the loss is the mean absolute difference between
        the dates (`unchange`) or one minus it (`change`)."""
        import torch

        batch, channels, height, width = x1.shape
        flat_1 = x1.reshape(batch, channels, -1)
        flat_2 = x2.reshape(batch, channels, -1)
        n = width
        first = torch.randint(0, height * width, (n,), generator=generator, device="cpu").to(x1.device)
        second = torch.randint(0, height * width, (n,), generator=generator, device="cpu").to(x1.device)
        sim_1 = torch.nn.functional.cosine_similarity(flat_1[:, :, first], flat_1[:, :, second], dim=1)
        sim_2 = torch.nn.functional.cosine_similarity(flat_2[:, :, first], flat_2[:, :, second], dim=1)
        value = torch.mean(torch.abs(sim_1 - sim_2))
        return value if mode == "unchange" else 1.0 - value

    def _loss(self, change: Any, maps_1: Any, maps_2: Any, focuses: Any, target: Any, generator: Any) -> Any:
        """Upstream `Loss`: MSE between the tanh change map and the 0 / 1 target over the labelled pixels, plus
        `CONTENT_LOSS_WEIGHT / n` times the summed change- and unchange-content losses over the n scales."""
        import torch

        valid = target != IGNORE_INDEX
        main = torch.nn.functional.mse_loss(change[valid], target.float()[valid])
        content = change.new_zeros(())
        for map_1, map_2, focus in zip(maps_1, maps_2, focuses, strict=True):
            weight = focus.unsqueeze(1)
            content = content + self._content_loss(map_1 * weight, map_2 * weight, generator, mode="change")
            content = content + self._content_loss(map_1 * (1 - weight), map_2 * (1 - weight), generator, mode="unchange")
        return main + CONTENT_LOSS_WEIGHT / len(maps_1) * content

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 6,
        lr: float = 1e-4,
        batch_size: int = 4,
        trainable: str = "change_decoder",
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning of the change decoder (`trainable="change_decoder"`; `"decoders"` also unfreezes the
        two content decoders) on labelled pairs: the upstream loss (MSE on the change map plus the content terms),
        AdamW at a fixed learning rate, seeded horizontal/vertical flips applied to both dates and the label,
        float16 autocast with loss scaling on CUDA, BatchNorm statistics frozen. Epoch 0 records the frozen model;
        the epoch with the lowest validation loss is kept."""
        if not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not (0.0 < lr <= 1e-2):
            raise ValueError("lr must be in (0, 1e-2]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 16:
            raise ValueError("batch_size must be an int in 1..16")
        names = self._trainable(trainable)
        train_checked = validate_dataset(train)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None
        sizes = {tuple(r["before"].shape) for r in train_checked} | {tuple(r["before"].shape) for r in val_checked or []}
        if len(sizes) != 1:
            raise ValueError(f"adaptation needs pairs of one size, got {sorted(sizes)}")
        import numpy as np
        import torch

        torch.manual_seed(seed)
        generator = torch.Generator(device="cpu").manual_seed(seed)
        started = time.perf_counter()
        model = self.model
        name_set = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in name_set)
        params = [p for n, p in model.named_parameters() if n in name_set]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
        use_amp = self.device.startswith("cuda")
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        rng = np.random.default_rng(seed)

        def batch_loss(batch: list[dict[str, Any]], *, grad: bool) -> Any:
            before = np.stack([r["before"] for r in batch])
            after = np.stack([r["after"] for r in batch])
            labels = np.stack([r["label"] for r in batch])
            if grad:
                if rng.random() < 0.5:
                    before, after, labels = before[:, :, ::-1], after[:, :, ::-1], labels[:, :, ::-1]
                if rng.random() < 0.5:
                    before, after, labels = before[:, ::-1], after[:, ::-1], labels[:, ::-1]
            change, maps_1, maps_2, focuses = self._change_map(
                np.ascontiguousarray(before), np.ascontiguousarray(after), grad=grad, with_content=True
            )
            target = torch.from_numpy(np.ascontiguousarray(labels)).to(self.device)
            return self._loss(change, maps_1, maps_2, focuses, target, generator)

        def val_loss() -> float | None:
            if val_checked is None:
                return None
            model.eval()
            losses = []
            for start in range(0, len(val_checked), batch_size):
                losses.append(float(batch_loss(val_checked[start : start + batch_size], grad=False)))
            return sum(losses) / len(losses)

        initial_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
        try:
            history: list[dict[str, Any]] = []
            entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val_loss": val_loss(), "note": "frozen model"}
            if val_checked is not None:
                entry["val"] = self.evaluate(val_checked, batch_size=batch_size)["model"]
            history.append(entry)
            best_val = entry["val_loss"] if entry["val_loss"] is not None else math.inf
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
            best_epoch = 0
            if progress:
                progress(entry)
            n_steps = 0
            for epoch in range(1, epochs + 1):
                model.train()
                for module in model.modules():  # BatchNorm statistics stay frozen: tiny batches would corrupt them
                    if isinstance(module, torch.nn.modules.batchnorm._BatchNorm):
                        module.eval()
                order = rng.permutation(len(train_checked)).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    batch = [train_checked[i] for i in order[start : start + batch_size]]
                    loss = batch_loss(batch, grad=True)
                    optimiser.zero_grad(set_to_none=True)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimiser)
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    scaler.step(optimiser)
                    scaler.update()
                    losses.append(float(loss.detach()))
                    n_steps += 1
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val_loss": val_loss()}
                if val_checked is not None:
                    entry["val"] = self.evaluate(val_checked, batch_size=batch_size)["model"]
                history.append(entry)
                if progress:
                    progress(entry)
                if entry["val_loss"] is None or entry["val_loss"] < best_val:
                    best_val = entry["val_loss"] if entry["val_loss"] is not None else best_val
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in training, validation or the progress callback leaves the model as it
            # was before adapt() (trained tensors restored), frozen, with no adapter attached.
            restore = dict(model.state_dict())
            restore.update(initial_state)
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable": trainable,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "lr": lr,
            "batch_size": batch_size,
            "loss": "upstream loss: MSE on the tanh change map + 0.1 × the content-consistency terms, ignore index excluded",
            "augmentation": "seeded horizontal/vertical flips of both dates and the label",
            "batchnorm": "running statistics frozen (eval mode) during adaptation",
            "precision": "float16 autocast + GradScaler" if use_amp else "float32",
            "n_train_records": len(train_checked),
            "n_steps": n_steps,
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts -------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted tensors as safetensors with a manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in self.model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": MODEL_KEY, "converted_sha256": CONVERTED_SHA256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": _sha256_file(weights_path)}
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return out

    @staticmethod
    def check_artifact_manifest(root: Path, manifest: Mapping[str, Any]) -> tuple[Path, str]:
        """Static checks on an adapter manifest, before any model or weights work: format and version, the pinned
        base and converted digest, exactly one weights entry named `adapter.safetensors` inside the artifact
        directory, and an adaptation mode that is one of the declared scopes. Returns the weights path and mode."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not supported "
                f"(expected {ARTIFACT_FORMAT_VERSION!r})"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError("artifact was adapted from a different base model or revision")
        if base.get("converted_sha256") != CONVERTED_SHA256:
            raise ValueError("artifact records a different converted-base digest")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one weights file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact weights file must be named {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weights file must sit inside the artifact directory")
        adapter = manifest.get("adapter")
        mode = adapter.get("trainable") if isinstance(adapter, Mapping) else None
        if mode not in ADAPTATION_MODES:
            raise ValueError(f"artifact adapter.trainable must be one of {ADAPTATION_MODES}")
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path, mode

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, scope and digest, then overwrite exactly the tensors the scope allows."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path, mode = self.check_artifact_manifest(root, manifest)
        expected = self._trainable(mode)
        if sorted(manifest["tensors"]) != expected:
            raise ValueError(
                f"artifact tensor list does not match the {len(expected)} tensors that trainable={mode!r} may change"
            )
        entry = manifest["files"][0]
        if _sha256_file(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from the validated manifest")
        state = self.model.state_dict()
        for key, value in tensors.items():
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(f"artifact tensor {key} has shape {tuple(value.shape)}, model has {tuple(state[key].shape)}")
        merged = dict(state)
        merged.update({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()})
        self.model.load_state_dict(merged, strict=True)
        self.model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": manifest["tensors"], "history": manifest.get("history", [])}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        require_source: bool = True,
    ) -> CFNetChangePipeline:
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        cls.check_artifact_manifest(root, manifest)
        pipeline = cls.from_pretrained(
            device=device, weights_dir=weights_dir, allow_download=allow_download, require_source=require_source
        )
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 4/4:** `src/cfnet_change_detection_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled-pair dataset contract for adapting the change detector: the pinned LEVIR-CD sample, roles from the
dataset's own splits, BYOD loaders and sample export.

The default dataset is **real**: 64 labelled 256 × 256 crops of LEVIR-CD (Chen and Shi, 2020) — pairs of 0.5 m
Google Earth patches of Texas cities taken 5–14 years apart with building-change labels — as the CFNet authors
processed and mirrored them on the Hugging Face Hub (`wifibk/CFNet_Datasets`, `LEVIR-CD-processed.tar.gz`: 25
overlapping 256 × 256 crops of every 1024 × 1024 pair, 11,125 / 1,600 / 3,200 crops in the train / val / test
folders). The 64 crops were drawn on 2026-09-20 with a fixed seed from the crops whose label is a clean 0 / 255
mask with at least 3 % change — one crop per source pair, from 32 training pairs, 8 validation pairs and 24 test
pairs — so the roles are the dataset's published splits (the checkpoint was trained on the training split,
selected on the validation split and reported on the test split). The tarball is pinned by byte size and SHA-256,
each pinned member is pinned again by size and SHA-256 and extracted **without** `extractall` into the cache, and
everything else in the archive is left alone. The repository redistributes none of the images.

**LEVIR-CD's terms:** "All images and annotations in LEVIR-CD can only be used for academic purposes, but are
prohibited for any commercial use", and the imagery is subject to Google Earth's terms of use. The tutorial fetches
the authors' mirror at run time for that academic purpose and nothing else; a commercial deployment must bring its
own labelled pairs.

A record is ``{id, before, after, label}``: two (H, W, 3) uint8 RGB images (or PNG / JPEG paths) and an (H, W) mask
with 0 = unchanged, 1 = changed, -1 = no data (or a PNG path with 0 / 255).
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import tarfile
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import (` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "LEVIR-CD building-change crops (Chen and Shi, 2020), the CFNet authors' processed mirror"
CORPUS_RELEASE = (
    "Hugging Face dataset wifibk/CFNet_Datasets, LEVIR-CD-processed.tar.gz, 64 crops selected 2026-09-20 from the "
    "dataset's own splits"
)
CORPUS_LICENSE = "LEVIR-CD: academic purposes only, commercial use prohibited; imagery subject to Google Earth's terms of use"
DATASET_ID = "wifibk/CFNet_Datasets"
DATASET_REVISION = "ba68aa9a54ae15fe32ce9b02c380eb384fae528e"
CORPUS_BASE_URL = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/"
TAR_NAME = "LEVIR-CD-processed.tar.gz"
TAR_BYTES = 3_831_872_824
TAR_SHA256 = "6515dd451c159b9ed5bd53b3fb6e15188dd114b296ff9169ca21fbcabcd1d109"
CORPUS_BYTES = 16_921_921  # the 192 pinned members, uncompressed
DEFAULT_CACHE_DIR = Path("weights") / "levir-cd"
ROLES = ("train", "validation", "test")
# (crop key, role = the dataset split, source pair id, before member, bytes, sha256, after member, bytes, sha256,
#  label member, bytes, sha256) — member paths are relative to the tarball root (the archive prefixes them with `./`)
SAMPLE_RECORDS: tuple[tuple[str, str, int, str, int, str, str, int, str, str, int, str], ...] = (
    (
        "train_2_13",
        "train",
        2,
        "LEVIR-CD-processed/train/A/train_2_13.png",
        131373,
        "9101cf4d881d921e4fbb97f7ab17fc684fe8f5dbd15f7fc9448d973e405b1dc8",
        "LEVIR-CD-processed/train/B/train_2_13.png",
        139058,
        "de76943a524787fbf19264c3822e7c393f605d138af8c005bbcaea72a0d80b95",
        "LEVIR-CD-processed/train/label/train_2_13.png",
        2017,
        "a6f218f5b98517e70d07bb7d8a42776fd446c634234b02485e6444ec4de9fa5e",
    ),
    (
        "train_3_8",
        "train",
        3,
        "LEVIR-CD-processed/train/A/train_3_8.png",
        146642,
        "56fea1f62235b8a03594a09406658c5382a7b451bd396d51dd5bacceda3f3332",
        "LEVIR-CD-processed/train/B/train_3_8.png",
        135129,
        "4866e3f40162bcd083b85825881a7ad3dbb6cba99b0e7c4b47779f052a94443d",
        "LEVIR-CD-processed/train/label/train_3_8.png",
        1122,
        "7b4a6645859f2944003d321079023f1b212b50ca16648e2edb093db3e8723400",
    ),
    (
        "train_7_18",
        "train",
        7,
        "LEVIR-CD-processed/train/A/train_7_18.png",
        148943,
        "122222370c3e0d64245db936f7ef9c49e908672ef1e47db472800d946d3fda4d",
        "LEVIR-CD-processed/train/B/train_7_18.png",
        127318,
        "8b716e17736e18412e4666bea6a1313f3759c87014e7a027d2fcb0287f236f58",
        "LEVIR-CD-processed/train/label/train_7_18.png",
        490,
        "0ab9e39abdd27c0b1ebfd982de071901f6ba91569e64a583fbe97b9c634f0b0f",
    ),
    (
        "train_26_9",
        "train",
        26,
        "LEVIR-CD-processed/train/A/train_26_9.png",
        162404,
        "53cb963c8c8d62307394847733c3ecf93f6811dc780547cb6ea269f81ac45a45",
        "LEVIR-CD-processed/train/B/train_26_9.png",
        145061,
        "0eb5b8b6a1dc34f66f81bb3110cae07779d5defc3bdae6240b836f16a81cd76a",
        "LEVIR-CD-processed/train/label/train_26_9.png",
        1189,
        "c6f894e6fd7ea55ccd25cfdfc8175ef36ce12faec43e657d2848d84d5deefaa6",
    ),
    (
        "train_30_22",
        "train",
        30,
        "LEVIR-CD-processed/train/A/train_30_22.png",
        149046,
        "e3a0de96355906af020a8f9aca527f2fbb37080088f18f1074456ae489351425",
        "LEVIR-CD-processed/train/B/train_30_22.png",
        131261,
        "24c021cc1efd9555587a1ee95ff2a570c56c78c2e530f480ec13d060165253af",
        "LEVIR-CD-processed/train/label/train_30_22.png",
        560,
        "2f1a711ad3ad1d0affc18f2e49ca87c793889fcc8dfe721909aadfc15cd1d45b",
    ),
    (
        "train_37_23",
        "train",
        37,
        "LEVIR-CD-processed/train/A/train_37_23.png",
        101625,
        "7da15823b2769bd9b4fdcb33024649b6c0e0b41b6d39e7ff9e6b3f44fa4b4514",
        "LEVIR-CD-processed/train/B/train_37_23.png",
        134786,
        "26249b35b50114abfb96342b49118195cb2bcc6b7958f6203a65f15744a13cde",
        "LEVIR-CD-processed/train/label/train_37_23.png",
        583,
        "5c952c2a100568538d702f1b5d581d7493ed26815bb623a7ee7639c92bad4567",
    ),
    (
        "train_42_0",
        "train",
        42,
        "LEVIR-CD-processed/train/A/train_42_0.png",
        118551,
        "746181a35206c07495c54349209dba9912da84ea8c200de98f417a20c0e45d62",
        "LEVIR-CD-processed/train/B/train_42_0.png",
        143430,
        "6afc6b6f8c74858608368d7506301f37ef4be4e4840f2a18247eabc2567fa5cd",
        "LEVIR-CD-processed/train/label/train_42_0.png",
        513,
        "c2c52111e9d56b798d8d0086baf1ecc2c3e172252e9ee08a4abf95d3e3aa4973",
    ),
    (
        "train_51_15",
        "train",
        51,
        "LEVIR-CD-processed/train/A/train_51_15.png",
        162074,
        "f91d7a962b04cf6739d31d8dbb6d5d9d7ae00bb7c0bb519f8165bfae8cbcd095",
        "LEVIR-CD-processed/train/B/train_51_15.png",
        130651,
        "995c408511588fa2db0cf8d97eec714dbf8fd0ea6e315412fc47b73eef35a601",
        "LEVIR-CD-processed/train/label/train_51_15.png",
        895,
        "9d1a01fc3e786e59d832d6e8250ef9dacffeb46cf1ba25bfb246a9d55a50a92d",
    ),
    (
        "train_84_11",
        "train",
        84,
        "LEVIR-CD-processed/train/A/train_84_11.png",
        170532,
        "daa84ea86ac5f2654f3954852822475bc57030b2501004b68a8880b832c19009",
        "LEVIR-CD-processed/train/B/train_84_11.png",
        136411,
        "bfd1adca5316e48bd315b634c1875daea47d272c4c740a87446e8d5a6460e71d",
        "LEVIR-CD-processed/train/label/train_84_11.png",
        553,
        "49fe1cb17c960ae494c5f25f9bc779aa37cd60bde1681a3c01050edb7ca5f10e",
    ),
    (
        "train_95_24",
        "train",
        95,
        "LEVIR-CD-processed/train/A/train_95_24.png",
        165684,
        "8a48cca61035b190374fbd2b1afb53229bad49686195be646707039d0c924871",
        "LEVIR-CD-processed/train/B/train_95_24.png",
        135304,
        "fc0f940d25ead08e4b3659616b002caee07e42261b1fba1616d3077e8fa4b319",
        "LEVIR-CD-processed/train/label/train_95_24.png",
        1178,
        "edebce2ef7eb9c4d394dd35e8e440e57fce547aeb7a8e3f389df522c01818acf",
    ),
    (
        "train_111_0",
        "train",
        111,
        "LEVIR-CD-processed/train/A/train_111_0.png",
        177762,
        "a7af358d0b07bf359836472a62aee644ff915cbd402a2c0a6fda6904f10a2176",
        "LEVIR-CD-processed/train/B/train_111_0.png",
        142613,
        "0f8742754a87dfba3f50f77effe95ad9b32524550f5784ad20ea26084814a2ff",
        "LEVIR-CD-processed/train/label/train_111_0.png",
        2800,
        "3c26e193ce0a1efbd6b1ea2b033e8a3d31f170b1f787698254065b9eb9df86e6",
    ),
    (
        "train_118_6",
        "train",
        118,
        "LEVIR-CD-processed/train/A/train_118_6.png",
        86352,
        "a66176c0560acd2d6f211f1ecdb030e258d3b1fe897a194bf4219c4abb2f6864",
        "LEVIR-CD-processed/train/B/train_118_6.png",
        143966,
        "6d8a6d3b77d73ff2ef4784c4e75344fe2fdc074a81cdd8dda85d469e9ec793d0",
        "LEVIR-CD-processed/train/label/train_118_6.png",
        570,
        "6f86c353f121058e61f6c8808d550ed18980f50eca12ab9718d901f9b633c24d",
    ),
    (
        "train_122_19",
        "train",
        122,
        "LEVIR-CD-processed/train/A/train_122_19.png",
        178952,
        "5471c60cc49ac0d21b011da7879c1c6457b6172ca2177154e4778914d8665dcc",
        "LEVIR-CD-processed/train/B/train_122_19.png",
        143933,
        "c60e85ceb9374bdc83433d66a604dce96b36435776385703ebd58eb68eafce06",
        "LEVIR-CD-processed/train/label/train_122_19.png",
        2020,
        "b59ef303d0bbf55d379f92872549a9c416d2fe31b6ff7b86b3a14c473a1f2eed",
    ),
    (
        "train_148_23",
        "train",
        148,
        "LEVIR-CD-processed/train/A/train_148_23.png",
        155771,
        "3d24dbbfe0acb6202363d4c2b797e2011c0c5dc941c56b702db5cdabaabcb2aa",
        "LEVIR-CD-processed/train/B/train_148_23.png",
        110069,
        "f8da3c1034d006c5785e444cf267c067b70dff7279323cffd566c4c10986cc59",
        "LEVIR-CD-processed/train/label/train_148_23.png",
        1019,
        "238af4de4e48939613e2495c252e4f4be61234821b40a088ea0387e24d35640f",
    ),
    (
        "train_149_2",
        "train",
        149,
        "LEVIR-CD-processed/train/A/train_149_2.png",
        159393,
        "a8ca407697d2ff2ff1b984378bab37d9a8a38ed5b498ed6bc57b550031b62a61",
        "LEVIR-CD-processed/train/B/train_149_2.png",
        119432,
        "960e713e035f59cf4abf3be70a31c342639b586a305615a96a579b748e5d423e",
        "LEVIR-CD-processed/train/label/train_149_2.png",
        1264,
        "55a235d7ac65e349e6e0234db4b9773fbac4f4ba7254301c13d0301afaba4d68",
    ),
    (
        "train_226_21",
        "train",
        226,
        "LEVIR-CD-processed/train/A/train_226_21.png",
        93171,
        "7c53678d2e924cdbeed0dfebd2f68f20ad342d3435992a41e96c84dc29bd1048",
        "LEVIR-CD-processed/train/B/train_226_21.png",
        119881,
        "fd7a18a9fa5006b82ef0467ef5f9643c1c0e7cc37f3812fd227901af46b4a702",
        "LEVIR-CD-processed/train/label/train_226_21.png",
        1705,
        "6ebbd79eb77b132c0a426012041a9a75179eb1d67a2f9b134f5059a54a3b6cf6",
    ),
    (
        "train_231_15",
        "train",
        231,
        "LEVIR-CD-processed/train/A/train_231_15.png",
        137114,
        "ab6e2255b27f6c2c380f316d184eabb416962717926c457d7490ce76ff1af812",
        "LEVIR-CD-processed/train/B/train_231_15.png",
        139562,
        "d950d9cec25b4e311ac648273aced2a42026580e6dcd5e7d3095cd07164777f5",
        "LEVIR-CD-processed/train/label/train_231_15.png",
        2464,
        "ebbeb30aa17e981af374821305e3b1297d5b5b3dec55899cccfece5687932937",
    ),
    (
        "train_233_4",
        "train",
        233,
        "LEVIR-CD-processed/train/A/train_233_4.png",
        95164,
        "ee6f9a0a01d9e9471e8582a3a09c465529c5bda09c9e8a611410c75e07b67c8a",
        "LEVIR-CD-processed/train/B/train_233_4.png",
        121028,
        "74b03a8e84946aea267e1525fe5e996f3fd06cb4f57c2aebde231df947d276c4",
        "LEVIR-CD-processed/train/label/train_233_4.png",
        1910,
        "521350b380c7b16de629fe91be293e388240270e3a1f390c75c4ee9b6c740c06",
    ),
    (
        "train_237_19",
        "train",
        237,
        "LEVIR-CD-processed/train/A/train_237_19.png",
        76131,
        "b2efeeac24d86a998fe450c76b60188b62ad7a20777fb11d97ca5e82467946ba",
        "LEVIR-CD-processed/train/B/train_237_19.png",
        82074,
        "c0c66e152342aeda2f661f2eae7d243b89e69215b3ea1f36a6acd0abcccf30e5",
        "LEVIR-CD-processed/train/label/train_237_19.png",
        649,
        "482310846c93dbd09565b504ad74466b1729180eedb37c926e33e87c27908871",
    ),
    (
        "train_299_10",
        "train",
        299,
        "LEVIR-CD-processed/train/A/train_299_10.png",
        127875,
        "ada3288f9dba51648d5ef9cdab12c521d0f6ac8649b3940c3a3e6d06175abade",
        "LEVIR-CD-processed/train/B/train_299_10.png",
        112039,
        "214993a029573864fc440cc388178460789057c2a2f52c608fea9a1179af05c7",
        "LEVIR-CD-processed/train/label/train_299_10.png",
        868,
        "12251b46f7a468c530b15a9ee58a140df0e077e548bbdd58de443552aa7f52b1",
    ),
    (
        "train_300_4",
        "train",
        300,
        "LEVIR-CD-processed/train/A/train_300_4.png",
        124209,
        "7858c2f17ab438b7485d37bf44325d9b222602e30a1a32c1ad5bdcea19c213f8",
        "LEVIR-CD-processed/train/B/train_300_4.png",
        101030,
        "0dfa72b1b5082e16f31cad700b6b87ac58a07fa10efc023858aa42553c56446f",
        "LEVIR-CD-processed/train/label/train_300_4.png",
        713,
        "39e9f31e0f54d1b8ca347740cd0e9c81a4fb415a47c3495ae97cfe38f8d12ec4",
    ),
    (
        "train_305_9",
        "train",
        305,
        "LEVIR-CD-processed/train/A/train_305_9.png",
        128725,
        "0253340459757b3f8839d01a59fda34898fcd6a7f83be181533a7dc1806ced1a",
        "LEVIR-CD-processed/train/B/train_305_9.png",
        132218,
        "f8977e86770c9eb2d698959a8023e41069bd8162ac857d969ddfe687b83e9a44",
        "LEVIR-CD-processed/train/label/train_305_9.png",
        2138,
        "4f587efb362137e989ab9b548a3b4f6a8d555cecf0c13ac997b78ab948524419",
    ),
    (
        "train_311_6",
        "train",
        311,
        "LEVIR-CD-processed/train/A/train_311_6.png",
        108147,
        "a9b1cdaee5b1670c82b30cbc638688490af5b0890fea3689afbe6c0bd0cce686",
        "LEVIR-CD-processed/train/B/train_311_6.png",
        107827,
        "10839b3eb6984a6a4592d5e19eca7dc24fe4669c38e5795a28524bcffda4d423",
        "LEVIR-CD-processed/train/label/train_311_6.png",
        523,
        "a04a7b82bbff4e85690dec9f5dc87b2ecaf35335fdb6e2f2a7849b0ac287bfb7",
    ),
    (
        "train_314_20",
        "train",
        314,
        "LEVIR-CD-processed/train/A/train_314_20.png",
        164138,
        "f9ac244f80c5259be068839a65c6981fd12afe46ea810eb46db0b6b180139ba0",
        "LEVIR-CD-processed/train/B/train_314_20.png",
        115166,
        "fb73a00e8f5eb9fb2a2bc92315dad913bb0ec4306fa86493873bee8c4ca8b276",
        "LEVIR-CD-processed/train/label/train_314_20.png",
        621,
        "60d82dd8446f0299e6cf64bf541a6fb8f050e6f35ffffe5896e829842edd2b91",
    ),
    (
        "train_346_1",
        "train",
        346,
        "LEVIR-CD-processed/train/A/train_346_1.png",
        146271,
        "4701c8519c48267ea06c61bee3b88a66f15ffd2c1b9f5e6a15a82b8d82e0b032",
        "LEVIR-CD-processed/train/B/train_346_1.png",
        129026,
        "1675df817f982f90ce0aecb73198c284e4ce567c7b438ca0bb6b70d71deaebcf",
        "LEVIR-CD-processed/train/label/train_346_1.png",
        725,
        "f5048d3f0a2adf8ce227a684efed7173ef1006ead78d0737fa920d1f14c49d3d",
    ),
    (
        "train_358_1",
        "train",
        358,
        "LEVIR-CD-processed/train/A/train_358_1.png",
        125015,
        "eaef39114aa8c025fbba056b3848ce50c60593e78f7ba471b8d3e0d7f5d9dffc",
        "LEVIR-CD-processed/train/B/train_358_1.png",
        112562,
        "ae2a21003846ce8b85b0ee629cc9195e4377ae87dc571b5a7e1cd829fc9d9148",
        "LEVIR-CD-processed/train/label/train_358_1.png",
        621,
        "498292479419bb2bb4147cb7ca063d3f4011fb251d3621e3dc7f7b180525cfc7",
    ),
    (
        "train_359_0",
        "train",
        359,
        "LEVIR-CD-processed/train/A/train_359_0.png",
        154413,
        "36bf727867420dd1f6237f8ee30833a46d5e660c4f8ce833eed9436e47be978e",
        "LEVIR-CD-processed/train/B/train_359_0.png",
        113445,
        "2f6a90370b6c1ccc8e34f640cd825a2e741e91b1be378b0d8a87be0af5bb2957",
        "LEVIR-CD-processed/train/label/train_359_0.png",
        289,
        "b8c8f0fa51bc69966c5dd816abd90585053d73764a9da18adc27924906f00c0e",
    ),
    (
        "train_362_9",
        "train",
        362,
        "LEVIR-CD-processed/train/A/train_362_9.png",
        96493,
        "60c3c4a62a54db3f0ba36c3da583ef51c6955bd061ababcd064a53f78e94b151",
        "LEVIR-CD-processed/train/B/train_362_9.png",
        112987,
        "1803f612476b41ef178493a239217ce27cc587e3a3bd4fc2ed1a5d7815d92814",
        "LEVIR-CD-processed/train/label/train_362_9.png",
        558,
        "557645e9b88e79824e1a181635d4e959c791f4e854545f2b7b0e1902b515e5f5",
    ),
    (
        "train_404_13",
        "train",
        404,
        "LEVIR-CD-processed/train/A/train_404_13.png",
        120659,
        "0e0a838ed3a260cb8be581094bc2a47e3497f54cc132ec108c7b97acab7224d9",
        "LEVIR-CD-processed/train/B/train_404_13.png",
        121388,
        "9bbd3f535634e35eb334dbabe9e3de2a985df19f1d894252bcf4cc8d51b5e4d9",
        "LEVIR-CD-processed/train/label/train_404_13.png",
        785,
        "b940c7e24ebd272a01c3f903ebc779f7865d531d9f2335bd2090d4d80398c993",
    ),
    (
        "train_414_10",
        "train",
        414,
        "LEVIR-CD-processed/train/A/train_414_10.png",
        90392,
        "0735afe057b19a64e9f8d4da460595f63377ebcf8c2ecbc02912c8f5901b5e58",
        "LEVIR-CD-processed/train/B/train_414_10.png",
        130745,
        "7ab753f944389aac65b5623fd89d40310c97da9ad68ccfa3960bdcc839020dfc",
        "LEVIR-CD-processed/train/label/train_414_10.png",
        1848,
        "cbbbf236d8b9a00c4d539b0b85ea20d2f904a56255f7c7e5a9edcd0eaf9308e2",
    ),
    (
        "train_425_9",
        "train",
        425,
        "LEVIR-CD-processed/train/A/train_425_9.png",
        95801,
        "e7c972f7bae56775ef7b20e93e5bc655bca63b22ebf5f1976a74093d9ff3bafb",
        "LEVIR-CD-processed/train/B/train_425_9.png",
        124006,
        "9f752a6b3f02e4220008f102ba044eb47fccd6692d3dc8cb7f69cb5bf28f6b81",
        "LEVIR-CD-processed/train/label/train_425_9.png",
        1833,
        "229cca6a374efaf812f3041820927d2c306b51b51768a426fd568d2fd3baf8d8",
    ),
    (
        "train_442_20",
        "train",
        442,
        "LEVIR-CD-processed/train/A/train_442_20.png",
        67918,
        "4af3eb2d979ba8b85241aced2265e95f39c25bc2b00c43b7e1cb771e38a524cb",
        "LEVIR-CD-processed/train/B/train_442_20.png",
        102805,
        "1d773004a5b13ee8f261fa23d85320a0339b1114c26e78fd4b8088defe108750",
        "LEVIR-CD-processed/train/label/train_442_20.png",
        1124,
        "3e503e368f3595f99b8d992a514e0e872856d5a995ccebeaf7983080e6945d65",
    ),
    (
        "val_6_24",
        "validation",
        6,
        "LEVIR-CD-processed/val/A/val_6_24.png",
        112061,
        "c99601cb38f4a4590d2ba7f00867a6d10390c9fbcce73f98233290c1af6d0abc",
        "LEVIR-CD-processed/val/B/val_6_24.png",
        145826,
        "eae0b0a56004df2d16ab77454ce123c515bf24fb1ddc38f4c818361c2f3be660",
        "LEVIR-CD-processed/val/label/val_6_24.png",
        2970,
        "4de2c752eb8ffefc0badd13e071b6de40d1418db06ef416859265d2f5398632a",
    ),
    (
        "val_13_9",
        "validation",
        13,
        "LEVIR-CD-processed/val/A/val_13_9.png",
        164058,
        "720c407b81695eb1b09163cddb1e0ce13165f38d0bf3e643d8f081a23034350e",
        "LEVIR-CD-processed/val/B/val_13_9.png",
        149631,
        "0e76fabab28ea263324d00d9eda95371cc9d39c47d3047e96aeabea26fcbb991",
        "LEVIR-CD-processed/val/label/val_13_9.png",
        896,
        "d85cab576ca4a8178092c41aa4927adc3bc6c81c00b258ad2da5932e07ad729e",
    ),
    (
        "val_34_5",
        "validation",
        34,
        "LEVIR-CD-processed/val/A/val_34_5.png",
        68368,
        "d4a8849b5110bd5a9efeeeb6f472279c018355809e19be3d2c47a62958eec24d",
        "LEVIR-CD-processed/val/B/val_34_5.png",
        140766,
        "c0aeafe424e614fdcefa84cc46a5f5731cc70204f34596a3dcd1d87e0cd68332",
        "LEVIR-CD-processed/val/label/val_34_5.png",
        3085,
        "bf0bbc96735ab9f8462ad0c0ba2eb1e36a8489f06601f2476bfb927e27ee54af",
    ),
    (
        "val_37_9",
        "validation",
        37,
        "LEVIR-CD-processed/val/A/val_37_9.png",
        162744,
        "d404664e468e74398f9c0088154e99dda8e7998f33107e5e7cf3eeb1692f90d5",
        "LEVIR-CD-processed/val/B/val_37_9.png",
        121061,
        "b959c466771b15ce39469e2a4645830f3d1775b715ac63b71cfd51010bf52769",
        "LEVIR-CD-processed/val/label/val_37_9.png",
        519,
        "e8b6ab1a26a571341589a869890bc43c9d0cbd0fea7467ed0c9df002347b9bad",
    ),
    (
        "val_39_16",
        "validation",
        39,
        "LEVIR-CD-processed/val/A/val_39_16.png",
        136529,
        "0b0fc77a3afd48341c2646518046fa6e87c75eff1a8ef464ccbe7f2bab273995",
        "LEVIR-CD-processed/val/B/val_39_16.png",
        132874,
        "225cd0cbbfc5e653be61ad332ff8578b4ac2bdc187d3cb2e5a2ffca928c69648",
        "LEVIR-CD-processed/val/label/val_39_16.png",
        1498,
        "ac32e38e175ab43aa31b318083e56203e52c7c0124f4378bc858543d69973f4d",
    ),
    (
        "val_40_15",
        "validation",
        40,
        "LEVIR-CD-processed/val/A/val_40_15.png",
        136860,
        "fef12f294e9f408371f64fa205766bc08ac9ee8187178834f2312c8b59b36816",
        "LEVIR-CD-processed/val/B/val_40_15.png",
        125469,
        "d30ce4e1e8a6350ef61bb6af8de0455f26985d6d135bcbf782acc435880ad08f",
        "LEVIR-CD-processed/val/label/val_40_15.png",
        1086,
        "19163c01367aa7b1caa2952ccd946fea1f9107c61169a5ba5a57ed11f1c063ee",
    ),
    (
        "val_42_5",
        "validation",
        42,
        "LEVIR-CD-processed/val/A/val_42_5.png",
        141920,
        "ce1807e780ad6a04b711f80ef96c8d6ae99de39c4d8ba51d69048a2bccc5d106",
        "LEVIR-CD-processed/val/B/val_42_5.png",
        103066,
        "440153679cf5159a70879c4a602decb49e9d51a0678a3bfb9940acfa295d4048",
        "LEVIR-CD-processed/val/label/val_42_5.png",
        513,
        "e84ba0e900478d275cf8d4ccbb18679c2193a7aa99df7f0cb499a1ccdcde681a",
    ),
    (
        "val_43_16",
        "validation",
        43,
        "LEVIR-CD-processed/val/A/val_43_16.png",
        131135,
        "5ff6bac90b14c45c467ad2bbaaaeedf47623d8aa516eb55079b6bea9b2e47ce5",
        "LEVIR-CD-processed/val/B/val_43_16.png",
        115808,
        "1ca82db4bc324eade98c0f0f337ef6c0e5efc59ee5b8a50e6f955277b16a8352",
        "LEVIR-CD-processed/val/label/val_43_16.png",
        1575,
        "15b5e948b2d085887afb9d26faec15f509ce66859241965cc74ad8c99afc5160",
    ),
    (
        "test_2_20",
        "test",
        2,
        "LEVIR-CD-processed/test/A/test_2_20.png",
        141865,
        "f7e0424b4b3b8dc7c2b0c3eb16cc3493b5676360314656c01014709fc744c4ce",
        "LEVIR-CD-processed/test/B/test_2_20.png",
        131326,
        "4cc3a90d556c66a92e3a8750b7e52523b0001cdf7d6c34ecdb9104d1053121ff",
        "LEVIR-CD-processed/test/label/test_2_20.png",
        671,
        "d8dab0716aed9fa32d3b39a8af5e955a96909709ad9dfc7feba576322e064a9e",
    ),
    (
        "test_3_19",
        "test",
        3,
        "LEVIR-CD-processed/test/A/test_3_19.png",
        145215,
        "cb4dc060d4cbd5d10decd84d0eb03b94f533cf59dba47f45f59eeee73eec57fe",
        "LEVIR-CD-processed/test/B/test_3_19.png",
        141296,
        "456e0b0900414f703720b9eb9b341423e368e7fbd8f57ede0491f171d6b29640",
        "LEVIR-CD-processed/test/label/test_3_19.png",
        654,
        "cfeb5522eb0530f56ee3b172d7d5597355e7f9ec7691bc0b76dd4251ca2ab21e",
    ),
    (
        "test_4_19",
        "test",
        4,
        "LEVIR-CD-processed/test/A/test_4_19.png",
        137794,
        "11bbfbe833c59c003b2833b6d1a8e7569e26a6b19d4b8681c157df0cc83c75fe",
        "LEVIR-CD-processed/test/B/test_4_19.png",
        122359,
        "358fa76346fc13ce816cf21e52171eae98bcec0abbfd882611fb6fb6b695bd84",
        "LEVIR-CD-processed/test/label/test_4_19.png",
        696,
        "4f624a332fc2533f2b512887ffb117ebe2a9cc87d1abb830bca1fde448f82810",
    ),
    (
        "test_8_14",
        "test",
        8,
        "LEVIR-CD-processed/test/A/test_8_14.png",
        153854,
        "7d07053d258dfadaf70e855b31d2cb63c633a88e9bf7907763c3fc3d4dd81f04",
        "LEVIR-CD-processed/test/B/test_8_14.png",
        135059,
        "2cfca66e9eaf33f3364efefb95902a6bd8b431db8cfedc75ef7c62578c4b3ffa",
        "LEVIR-CD-processed/test/label/test_8_14.png",
        2137,
        "c96a15c5c6c61659aa6fb10ed5f6fed50a7c4aea309e5a96da51efb4315115ee",
    ),
    (
        "test_9_2",
        "test",
        9,
        "LEVIR-CD-processed/test/A/test_9_2.png",
        135724,
        "7451cb17c2c51c1f9aeeb4d47be036a5a1dab760f983f2419986543697ed3b6f",
        "LEVIR-CD-processed/test/B/test_9_2.png",
        136605,
        "0b88777bfe60ad1fb68716102830b45c9408b351592b0455399fe5734c026808",
        "LEVIR-CD-processed/test/label/test_9_2.png",
        625,
        "2f84f033c2297ee17a29a84309672e1670f085231b0479a7c2c670f9057865ed",
    ),
    (
        "test_10_8",
        "test",
        10,
        "LEVIR-CD-processed/test/A/test_10_8.png",
        93161,
        "5b82e3fd3c35beae8b24a9c3934d68b4c1439dcc1c5ada90b1f0592b514e1068",
        "LEVIR-CD-processed/test/B/test_10_8.png",
        141767,
        "2403fdc75bfe267ee0f369229c943cfbe4da8f58416c1bf4eff56e3069816431",
        "LEVIR-CD-processed/test/label/test_10_8.png",
        1988,
        "e518a9651321ba95036fc8b0d7151bdd5e93f624059561a418633b0eedda4791",
    ),
    (
        "test_20_5",
        "test",
        20,
        "LEVIR-CD-processed/test/A/test_20_5.png",
        150549,
        "6c1eaa4030376693128db5f8cdf215308853b83ef7dbb5cfcd54f414b2955578",
        "LEVIR-CD-processed/test/B/test_20_5.png",
        146915,
        "452b474418576900a9d9d36b20107c8ee6cdf9fc2f713ca918eee66c05db8979",
        "LEVIR-CD-processed/test/label/test_20_5.png",
        2399,
        "19768846d74a9f8754d6d00d4257bbaf5e62e319002fe65bb4430c98029bd545",
    ),
    (
        "test_23_1",
        "test",
        23,
        "LEVIR-CD-processed/test/A/test_23_1.png",
        165292,
        "c60bbc22f9998c825dfc6db05fa34fb05edbe118308ffb257d0d20e9c2330daf",
        "LEVIR-CD-processed/test/B/test_23_1.png",
        141676,
        "4cba47cdcd533a59d412a98c9c6b1910cced3c94b670b822caeaefa03ff8d4d1",
        "LEVIR-CD-processed/test/label/test_23_1.png",
        1288,
        "7643b44c259c6e06434f683a5b664c68e34d2f6c2468f52593eb2033faf58223",
    ),
    (
        "test_28_2",
        "test",
        28,
        "LEVIR-CD-processed/test/A/test_28_2.png",
        164834,
        "67d73603c0704a331f377513eeca9f508f66cf0490c6df22a1596bccd563af20",
        "LEVIR-CD-processed/test/B/test_28_2.png",
        146091,
        "e4edadee584b74c0aaddc38be2d7805681efc2770b4d91896f1be2a38bf7c1da",
        "LEVIR-CD-processed/test/label/test_28_2.png",
        2182,
        "bfd31504e625c777f7da022b47f3c0b4129d089a98e2539b59f03d71fc430b2b",
    ),
    (
        "test_39_13",
        "test",
        39,
        "LEVIR-CD-processed/test/A/test_39_13.png",
        147482,
        "1638b941958e84e6d842b83129f7dfbec7c55bd6969f4ad99f39367ed1fa0e41",
        "LEVIR-CD-processed/test/B/test_39_13.png",
        110132,
        "5e023b21e986a79f996fd99f9ced0b3a7b680cf38645b95b0dd7e1fa5f9a9701",
        "LEVIR-CD-processed/test/label/test_39_13.png",
        1546,
        "44ac4b77080946a989b8c614adafc8366508f3776838ac8e175d543638e4619f",
    ),
    (
        "test_40_5",
        "test",
        40,
        "LEVIR-CD-processed/test/A/test_40_5.png",
        132757,
        "0378c28073526d476b6b5f4ca6d5e2f0d36d0e97feb787e531859389aa77d4fe",
        "LEVIR-CD-processed/test/B/test_40_5.png",
        133560,
        "a07fd9e242edd035d007e878e23211985c8068515885dc59357b4b98d11a15ab",
        "LEVIR-CD-processed/test/label/test_40_5.png",
        1857,
        "4515243885a055f616254a4fcbb58e6a5b783e1a42e8f7dbdaabcd2f1e151013",
    ),
    (
        "test_45_10",
        "test",
        45,
        "LEVIR-CD-processed/test/A/test_45_10.png",
        153525,
        "c7d7ab34df06a94b6dda3c5f37e209736c79fc567b777276c39770352de3ec4e",
        "LEVIR-CD-processed/test/B/test_45_10.png",
        138345,
        "cf772c4ed819ca5b56f3cf0ce16c1da89dffa892a47b2aed1f71f78405524da8",
        "LEVIR-CD-processed/test/label/test_45_10.png",
        3240,
        "2ebbeb40a0ad4a58ad537b73f244c301ff9285abd8f191cb31f53ba28d88edf6",
    ),
    (
        "test_46_11",
        "test",
        46,
        "LEVIR-CD-processed/test/A/test_46_11.png",
        157443,
        "9c554db0ca372ab0a7735fbe5b19960bbc9efbcb54bfe0c06fc341b8c8a20863",
        "LEVIR-CD-processed/test/B/test_46_11.png",
        144440,
        "9ef7c509d70ed8e12e740694a2333df7a1c2909d66c93c8e7227f2dcd76466d7",
        "LEVIR-CD-processed/test/label/test_46_11.png",
        3139,
        "c8817bd021f7f701a7eda3509ce05ac41d5a897804cda2ae3305436308314ffb",
    ),
    (
        "test_53_9",
        "test",
        53,
        "LEVIR-CD-processed/test/A/test_53_9.png",
        140640,
        "579903243f14d8473101dcdc0aef1bc6890bbf877eb039e63874f103b900c9d3",
        "LEVIR-CD-processed/test/B/test_53_9.png",
        145631,
        "f63d62f6c2b15f6a91d866947c4eeec77734ca8e6eb35d8c41646282cbb7265b",
        "LEVIR-CD-processed/test/label/test_53_9.png",
        844,
        "ea1a364a59d658523373e0607fdbdc7cff2ac05b9a66bf53e105e4f25eeb9a26",
    ),
    (
        "test_54_6",
        "test",
        54,
        "LEVIR-CD-processed/test/A/test_54_6.png",
        101916,
        "c8daf01f0b29fd90997fb41c51a7702b58dce8773165e460466f265b6ab50be3",
        "LEVIR-CD-processed/test/B/test_54_6.png",
        134302,
        "b4c8d761ddf5a2106397ce33e97dc45db10946495a9faabfeab27ac4c9f17255",
        "LEVIR-CD-processed/test/label/test_54_6.png",
        1206,
        "486c67f0b45037d52b6ae07c4eafe812dd5aff9c8c3e7dd0ddb404df4f38e0e0",
    ),
    (
        "test_72_15",
        "test",
        72,
        "LEVIR-CD-processed/test/A/test_72_15.png",
        74043,
        "50f30d1789bea1662e127b0ca69d93d4a4ce425d4142aa43fe7f88797a3888f3",
        "LEVIR-CD-processed/test/B/test_72_15.png",
        130279,
        "d56981df93439ff11721f8496d653762bcb66c30ba86cbc0e130222902794ead",
        "LEVIR-CD-processed/test/label/test_72_15.png",
        2579,
        "806ece0e09614920b0e98366e732b2bf1b332fb62494d22f98407a551d1d1231",
    ),
    (
        "test_74_0",
        "test",
        74,
        "LEVIR-CD-processed/test/A/test_74_0.png",
        124484,
        "4da5be567461bc44eab5f6aa2fbd6949140bf8eb04d8effd056d17b1c82a0772",
        "LEVIR-CD-processed/test/B/test_74_0.png",
        113788,
        "9484c2307f94c8c7f90f96f294dd97470a8b01ab10398a465305131825dedd94",
        "LEVIR-CD-processed/test/label/test_74_0.png",
        941,
        "ea74105f94f99ed51d602e138bc5d684bd8be792cacda39fe144af3b537b1a2f",
    ),
    (
        "test_76_22",
        "test",
        76,
        "LEVIR-CD-processed/test/A/test_76_22.png",
        172511,
        "1be2a8e4ba0c78975af8a3a0034d682823fa2bbf7a3fc4f5232377b851d12efc",
        "LEVIR-CD-processed/test/B/test_76_22.png",
        137754,
        "d8e950323a3b442371b91ab34cc446dfde240839959cee69c94c6db1ee5cad58",
        "LEVIR-CD-processed/test/label/test_76_22.png",
        466,
        "7dd0477543df99f94273ad512af901349face92c61a16dc0e4fd90dec2c3d856",
    ),
    (
        "test_79_15",
        "test",
        79,
        "LEVIR-CD-processed/test/A/test_79_15.png",
        161096,
        "adea19ea0168f518e79936e8ff1eaed05146bad80a29b34d7bc930816f2b7fb9",
        "LEVIR-CD-processed/test/B/test_79_15.png",
        124633,
        "fc8d0e572c8229cc2bd7d251666c6cda4e3f09aac7d0c09cbdb56ea2e1127f6a",
        "LEVIR-CD-processed/test/label/test_79_15.png",
        394,
        "5231f329ebcec7e74d07712eb81a5e6bc433835534a2a0a3beda8a15a2f04451",
    ),
    (
        "test_100_24",
        "test",
        100,
        "LEVIR-CD-processed/test/A/test_100_24.png",
        145607,
        "8b5642d499d8d97d62b19ca8e4b3aec33d550df451a4bccf8cca39f8dd80fd52",
        "LEVIR-CD-processed/test/B/test_100_24.png",
        140629,
        "fa8fa44dc7da0179d91b36aa406ee2285f75094e90835bfaa9cfff4674bd1e79",
        "LEVIR-CD-processed/test/label/test_100_24.png",
        2854,
        "1a37972c68b2803be80685c45710466fddbc6c78802c64bc2b33d64690549ed5",
    ),
    (
        "test_101_22",
        "test",
        101,
        "LEVIR-CD-processed/test/A/test_101_22.png",
        110280,
        "f5389f31868b635af8291a6fbeea17e4f31c2b6bfa86d976e33d5ff9c55b6221",
        "LEVIR-CD-processed/test/B/test_101_22.png",
        141281,
        "5c4adc15a1af7b4319ccd81be827d4302c2756c69485603f9e7faf8835bfe7b7",
        "LEVIR-CD-processed/test/label/test_101_22.png",
        854,
        "885b7450cc6ac965b6a9a541cfd525e8e66dc9f9eb5b810333ad9a1e6e87b45e",
    ),
    (
        "test_115_3",
        "test",
        115,
        "LEVIR-CD-processed/test/A/test_115_3.png",
        154916,
        "e7328316dc10f2f2a592e7d1a0f8c767318fc245544d0e367c4f48b408fe9e2a",
        "LEVIR-CD-processed/test/B/test_115_3.png",
        125364,
        "11e124e2b9369bb7b63464cd17b22d3b8e908ba80bddfeaa4cd587c64fcf7d9c",
        "LEVIR-CD-processed/test/label/test_115_3.png",
        659,
        "71412fc144315b89aa4777a034fdc53d5c35e0a0d27a29872ef48c298e26548d",
    ),
    (
        "test_118_3",
        "test",
        118,
        "LEVIR-CD-processed/test/A/test_118_3.png",
        169357,
        "db3cd0568118dde3ac7a92161d07813807249bebc387b290d408413f3fc9042b",
        "LEVIR-CD-processed/test/B/test_118_3.png",
        141099,
        "78beb206e121989f50182899f29e392efb9b3ade8797d4fde3f2207120a6d6b7",
        "LEVIR-CD-processed/test/label/test_118_3.png",
        2413,
        "74dc6d03c847b7d9c4ff1a1d000070f6d4ccd8294cbaa001dd66c4fd25f2388e",
    ),
    (
        "test_119_21",
        "test",
        119,
        "LEVIR-CD-processed/test/A/test_119_21.png",
        136388,
        "db596e11eb93487868b0189b718126f5c4989e069917f1756e474a9c030629f2",
        "LEVIR-CD-processed/test/B/test_119_21.png",
        111422,
        "fa61de13dc121afafe80c259a952c10c2cabeb89b44285ec97f01004d119063c",
        "LEVIR-CD-processed/test/label/test_119_21.png",
        387,
        "28f4691d040d849dca4b7b70c9251f7faa07ffe0244c1431edda5512d225569c",
    ),
)

SAMPLE_LABEL_SOURCE = f"{CORPUS_NAME}; {CORPUS_RELEASE}; {CORPUS_LICENSE}"


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 22), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _pinned_members() -> dict[str, tuple[int, str]]:
    out = {}
    for record in SAMPLE_RECORDS:
        for offset in (3, 6, 9):
            out[record[offset]] = (record[offset + 1], record[offset + 2])
    return out


def _hub_download_tarball(destination: Path) -> None:
    from huggingface_hub import hf_hub_download

    hf_hub_download(DATASET_ID, TAR_NAME, repo_type="dataset", revision=DATASET_REVISION, local_dir=str(destination.parent))


def fetch_tarball(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> Path:
    """The pinned dataset tarball in the cache, fetched from the Hub at the immutable revision when absent, and
    refused on a size or SHA-256 mismatch (the 3.8 GB file is hashed once per call)."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    local = cache / TAR_NAME
    if not local.is_file() or local.stat().st_size != TAR_BYTES:
        if fetcher is not None:
            local.write_bytes(fetcher(CORPUS_BASE_URL + TAR_NAME))
        else:
            _hub_download_tarball(local)
    size = local.stat().st_size
    digest = _sha256_file(local)
    if size != TAR_BYTES or digest != TAR_SHA256:
        raise ValueError(f"{TAR_NAME}: {size} bytes with sha256 {digest[:16]}…, pinned {TAR_BYTES} / {TAR_SHA256[:16]}…")
    return local


def _member_name(name: str) -> str:
    return name[2:] if name.startswith("./") else name


def extract_pinned_members(tar_path: str | Path, *, cache_dir: str | Path | None = None) -> dict[str, bytes]:
    """Stream through the tarball once and copy out exactly the pinned members (no `extractall`, no paths from
    the archive: each is written under `<split>_<folder>_<name>` in `cache_dir/crops/`), refusing a size or
    digest mismatch."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    crops = cache / "crops"
    crops.mkdir(parents=True, exist_ok=True)
    wanted = _pinned_members()
    out: dict[str, bytes] = {}
    with tarfile.open(tar_path, "r:gz") as archive:
        for member in archive:
            name = _member_name(member.name)
            if name not in wanted or not member.isfile():
                continue
            size, sha = wanted[name]
            handle = archive.extractfile(member)
            data = handle.read() if handle is not None else b""
            if len(data) != size or _sha256_bytes(data) != sha:
                raise ValueError(
                    f"{name}: {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, pinned {size} / {sha[:16]}…"
                )
            (crops / _cache_name(name)).write_bytes(data)
            out[name] = data
            if len(out) == len(wanted):
                break
    missing = sorted(set(wanted) - set(out))
    if missing:
        raise ValueError(f"tarball does not contain {len(missing)} pinned members, e.g. {missing[:3]}")
    return out


def _cache_name(member: str) -> str:
    """`LEVIR-CD-processed/test/A/test_1_2.png` -> `test_A_test_1_2.png` (flat, no archive paths)."""
    parts = member.split("/")
    return "_".join(parts[-3:])


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, dict[str, bytes]]:
    """Every pinned crop's before / after / label bytes, keyed by crop key: from the extracted cache when every
    file is present with its pinned digest, otherwise from the (verified) tarball."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    crops = cache / "crops"
    wanted = _pinned_members()
    cached: dict[str, bytes] = {}
    for member, (size, sha) in wanted.items():
        local = crops / _cache_name(member)
        if local.is_file() and local.stat().st_size == size:
            data = local.read_bytes()
            if _sha256_bytes(data) == sha:
                cached[member] = data
    if len(cached) != len(wanted):
        cached = extract_pinned_members(fetch_tarball(cache_dir=cache, fetcher=fetcher), cache_dir=cache)
    return {r[0]: {"before": cached[r[3]], "after": cached[r[6]], "label": cached[r[9]]} for r in SAMPLE_RECORDS}


def read_corpus(files: Mapping[str, Mapping[str, bytes]]) -> dict[str, list[dict[str, Any]]]:
    """Decode the verified bytes into `{id, before, after, label}` records grouped by role (train / validation /
    test)."""
    import tempfile

    splits: dict[str, list[dict[str, Any]]] = {role: [] for role in ROLES}
    for key, role, pair, before_member, *_rest in SAMPLE_RECORDS:
        if key not in files:
            raise ValueError(f"corpus is missing {key}")
        with tempfile.TemporaryDirectory() as tmp:
            paths = {}
            for part in ("before", "after", "label"):
                paths[part] = Path(tmp) / f"{part}.png"
                paths[part].write_bytes(files[key][part])
            before = read_image(paths["before"])
            after = read_image(paths["after"])
            label = read_mask(paths["label"])
        raw = {
            "id": f"{role}-{len(splits[role]):03d}",
            "source_id": key,
            "region": f"{role}-pair-{pair}",  # the 1024 × 1024 source pair the crop was cut from
            "split": role,
            "before": before,
            "after": after,
            "label": label,
            "source": f"{CORPUS_BASE_URL}{TAR_NAME}#{before_member}",
        }
        splits[role].append(check_record(raw))
    return splits


def fetch_sample_dataset(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus (roles = the dataset's own splits)."""
    return read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no pair (by pixel digest) and no source pair (by `region`) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    regions: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = pair_digest(record)
            if key in seen and seen[key] != name:
                raise ValueError(f"pair {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
            region = record.get("region")
            if region:
                if region in regions and regions[region] != name:
                    raise ValueError(f"source pair {region!r} has crops in both {regions[region]} and {name}")
                regions[region] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.25,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train / validation / test after de-duplicating pairs. Crops of one
    scene are near-duplicates; group them yourself (one scene per split) when that matters."""
    import random

    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = pair_digest(record)
        if key not in seen:
            seen.add(key)
            unique.append(record)
    rng = random.Random(seed)
    rng.shuffle(unique)
    n_test = max(1, round(len(unique) * test_fraction))
    n_val = round(len(unique) * val_fraction)
    splits = {"test": unique[:n_test], "validation": unique[n_test : n_test + n_val], "train": unique[n_test + n_val :]}
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(f"split leaves {len(splits['train'])} training pairs; at least {MIN_RECORDS} are required")
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, before, after, label}` records from a directory or a zip holding `pairs.csv` (columns `id`,
    `before`, `after`, `label`) beside same-sized RGB PNG / JPEG images and 0 / 255 label PNGs; files are decoded
    from bytes, never extracted to disk."""
    import tempfile

    source = Path(path)
    if source.is_dir():
        table = (source / "pairs.csv").read_text(encoding="utf-8")
        loader = lambda name: (source / name).read_bytes()  # noqa: E731
    elif source.is_file() and source.suffix.lower() == ".zip":
        archive = zipfile.ZipFile(source)
        members = {Path(n).name: n for n in archive.namelist()}
        if "pairs.csv" not in members:
            raise ValueError("BYOD zip must contain pairs.csv")
        table = archive.read(members["pairs.csv"]).decode("utf-8")
        loader = lambda name: archive.read(members[name])  # noqa: E731
    else:
        raise ValueError("BYOD datasets must be a directory or a .zip holding pairs.csv and the image files")
    rows = list(csv.DictReader(io.StringIO(table)))
    missing = {"id", "before", "after", "label"} - set(rows[0].keys() if rows else set())
    if missing:
        raise ValueError(f"pairs.csv is missing columns {sorted(missing)}")
    out = []
    with tempfile.TemporaryDirectory() as tmp:
        for row in rows:
            record: dict[str, Any] = {"id": row["id"]}
            for part in ("before", "after"):
                image_path = Path(tmp) / f"{part}{Path(row[part]).suffix.lower() or '.png'}"
                image_path.write_bytes(loader(row[part]))
                record[part] = read_image(image_path)
            if row.get("label"):
                label_path = Path(tmp) / "label.png"
                label_path.write_bytes(loader(row["label"]))
                record["label"] = read_mask(label_path)
            out.append(record)
    return out


def write_sample_pair(
    record: Mapping[str, Any], before_path: str | Path, after_path: str | Path, label_path: str | Path
) -> dict[str, str]:
    """Write one record as two RGB PNGs and a 0 / 255 label PNG (the BYOD shape) and return the paths."""
    import numpy as np
    from PIL import Image

    paths = {"before": Path(before_path), "after": Path(after_path), "label": Path(label_path)}
    paths["before"].parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(np.asarray(record["before"], dtype=np.uint8)).save(paths["before"])
    Image.fromarray(np.asarray(record["after"], dtype=np.uint8)).save(paths["after"])
    label = np.asarray(record["label"], dtype=np.int64)
    Image.fromarray(np.where(label == 1, 255, 0).astype(np.uint8)).save(paths["label"])
    return {k: str(v) for k, v in paths.items()}


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the pairs table of a split (id, before, after, label, provenance) in the shape BYOD expects."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "before", "after", "label", "region", "source"])
        writer.writeheader()
        for record in records:
            stem = record.get("source_id", record["id"])
            writer.writerow(
                {
                    "id": record["id"],
                    "before": f"{stem}_A.png",
                    "after": f"{stem}_B.png",
                    "label": f"{stem}_label.png",
                    "region": record.get("region", ""),
                    "source": record.get("source", ""),
                }
            )
    return out


def dataset_manifest(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Validate every split and summarise the dataset (counts, change balance, digests) for provenance exports."""
    summary: dict[str, Any] = {"model_id": MODEL_ID, "sample_size": SAMPLE_SIZE, "splits": {}}
    for name, records in splits.items():
        report = validate_dataset(records, min_records=1)
        summary["splits"][name] = {
            "n_records": report["n_records"],
            "sizes": report["sizes"],
            "change_fraction": report["change_fraction"],
            "ignored_pixels": report["ignored_pixels"],
            "regions": sorted({str(r.get("region", "")) for r in records if r.get("region")}),
            "digest": report["digest"],
        }
    summary["disjoint"] = check_split_disjoint(splits)
    digests = json.dumps({k: v["digest"] for k, v in summary["splits"].items()}, sort_keys=True)
    summary["digest"] = hashlib.sha256(digests.encode("utf-8")).hexdigest()
    return summary

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `2`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `c23279428d14…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `CFNetChangePipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), report=print)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "cfnet-levir-cd",
  "modelId": "wifibk/CFNet",
  "revision": "c23279428d14186d67ce199b3db358038bf37585",
  "files": [
    {
      "path": "levir-cd.pth",
      "bytes": 15805666,
      "sha256": "22ab286b2138ab1082e04290b27cbff5abd3802375a6b2f01d4ba78ca8b0c1aa"
    },
    {
      "path": "README.md",
      "bytes": 2974,
      "sha256": "5b8d54988a30b5ff819d7b9b6351c82dba253b8ee529f55dc79603657192c48b"
    }
  ],
  "totalBytes": 15808640
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = CFNetChangePipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), report=print)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Sample pairs, validation and roles

The default dataset is 64 labelled 256 × 256 crops of LEVIR-CD as the CFNet authors processed them (25 overlapping crops of every 1024 × 1024 pair): one crop from each of 32 training pairs, 8 validation pairs and 24 test pairs, drawn with a fixed seed from the crops whose label is a clean 0 / 255 mask with at least 3 % change, so every crop can be scored. Roles are the dataset's own splits — the checkpoint was trained on the training split, selected on the validation split and reported on the test split. `fetch_corpus` downloads the tarball from the Hub at its immutable revision, refuses it on any size or SHA-256 mismatch, streams through it once and copies out exactly the 192 pinned members — each refused on its own size or digest mismatch and written under a flat name, never at a path taken from the archive — then reads the RGB PNGs and the masks, which become 0 / 1. `dataset_manifest` validates every split, checks that no crop and no source pair appears twice and records a digest.

Look for: 32 / 8 / 24 crops with change fractions around 0.10–0.14, one source pair per crop, a written sample pair (`outputs/cfnet_change_detection_sample_before.png`, `_sample_after.png`, `_sample_label.png` — the BYOD shape), and three refusal probes — dates of different sizes, a side that is not a multiple of 32, a mask with an unknown value — each rejected before the model runs. The tarball takes a few minutes to fetch and about a minute to stream.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np

USE_BYOD = False  # @param {type:"boolean"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    splits = split_dataset(load_byod_dataset(byod_path), seed=0)
    data_source = 'BYOD (' + file_name + ')'
else:
    splits = fetch_sample_dataset(cache_dir='weights/levir-cd')
    data_source = SAMPLE_LABEL_SOURCE
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']

dataset_report = dataset_manifest({'train': train_records, 'validation': val_records, 'test': test_records})
print({'data_source': data_source, 'splits': {k: v['n_records'] for k, v in dataset_report['splits'].items()}, 'disjoint': dataset_report['disjoint'], 'digest': dataset_report['digest'][:16] + '...'})
for name, part in dataset_report['splits'].items():
    print({name: {'change_fraction': part['change_fraction'], 'sizes': part['sizes'], 'ignored_pixels': part['ignored_pixels'], 'source_pairs': len(part['regions'])}})
print({'first_test_pair': validate_inputs(test_records[0])})
sample_pair = write_sample_pair(test_records[0], 'outputs/cfnet_change_detection_sample_before.png', 'outputs/cfnet_change_detection_sample_after.png', 'outputs/cfnet_change_detection_sample_label.png')
print({'sample_pair': sample_pair, 'pairs_csv': str(write_dataset_csv(test_records, 'outputs/cfnet_change_detection_sample_pairs.csv'))})

print({'validation': INPUT_SCHEMA['validation']})
probes = {
    'dates of different sizes': [{**test_records[0], 'after': test_records[0]['after'][:128, :128]}, *test_records[1:4]],
    'side not a multiple of 32': [{**test_records[0], 'before': test_records[0]['before'][:200, :200], 'after': test_records[0]['after'][:200, :200], 'label': test_records[0]['label'][:200, :200]}, *test_records[1:4]],
    'unknown label value': [{**test_records[0], 'label': np.where(test_records[0]['label'] == 1, 2, test_records[0]['label'])}, *test_records[1:4]],
}
for name, records in probes.items():
    try:
        validate_dataset(records)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. The frozen model against the all-unchanged baseline

`pipe.predict` reorders each image to BGR, divides by 255 and standardises each date with its own LEVIR-CD statistics — exactly the upstream loader's preprocessing — runs the shared encoder, the two content decoders, the focuser and the change decoder (in float16 autocast on a GPU), and returns the change map (the network's `tanh` output in (−1, 1), not a calibrated probability), the binary mask (changed where the map exceeds 0.5, the upstream rule) and the changed fraction per pair. `pipe.evaluate` pools the labelled pixels of every held-out pair into one confusion matrix (−1 pixels excluded) and reports the F1, IoU, precision and recall of the changed class exactly as the upstream evaluation does, plus the overall accuracy; the **all-unchanged baseline** — every pixel predicted unchanged — is scored on the same pixels, so its accuracy is exactly the unchanged fraction and its F1 and IoU are 0.

Look for: a changed-class F1 near 0.93 and an IoU near 0.88 on the test crops (in the build record 0.9352 and 0.8782, precision 0.9424, recall 0.9281, against an all-unchanged accuracy of 0.8615 — the authors report F1 92.18 / IoU 85.49 on the full test split, whose crops are mostly change-free) and a validation F1 near 0.92. These are sample-sanity numbers on 24 crops chosen for having change, not the benchmark.

In [ ]:
import time

t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records)
frozen_val = pipe.evaluate(val_records)
print({'seconds': round(time.perf_counter() - t0, 1), 'metric': frozen_test['metric']})
print({'baseline_unchanged_test': {k: frozen_test['baseline_unchanged'][k] for k in ('f1', 'iou', 'accuracy')}})
print({'frozen_test': {k: frozen_test['model'][k] for k in ('f1', 'iou', 'precision', 'recall', 'accuracy', 'change_fraction_label', 'change_fraction_predicted')}})
print({'frozen_validation': {k: frozen_val['model'][k] for k in ('f1', 'iou')}})
frozen_predictions = pipe.predict(test_records)
for record, pred in list(zip(test_records, frozen_predictions['predictions']))[:6]:
    labelled = record['label'] >= 0
    print({'pair': record['source_id'], 'change_label': round(float((record['label'] == 1).sum() / labelled.sum()), 3), 'change_predicted': pred['changed_fraction'], 'map_range': [round(float(pred['change_map'].min()), 3), round(float(pred['change_map'].max()), 3)]})
print({'decision_rule': frozen_predictions['decision_rule'], 'map_shape': frozen_predictions['predictions'][0]['change_map'].shape})

## 6. Bounded fine-tuning of the change decoder

`pipe.adapt` trains the 132 tensors of the change decoder (852,867 parameters — 22 % of the model) and nothing else: the encoder and the two content decoders are frozen (no gradient is stored for them), and every BatchNorm layer keeps its running statistics, because batches of four crops would corrupt them. Each step takes four pairs with a seeded horizontal or vertical flip applied to both dates and the label, computes the upstream loss — the mean squared error between the `tanh` change map and the 0 / 1 target over the labelled pixels, plus 0.1 times the content-consistency terms that compare cosine similarities of random pixel pairs within each date's focused and unfocused content maps — and takes an AdamW step at a small fixed learning rate with gradient-norm clipping and float16 loss scaling. Epoch 0 records the frozen model's validation loss and metrics; the epoch with the lowest validation loss is kept — which can be epoch 0, since the packaged model already trained on this split.

Watch the validation loss: in the build record it fell from 0.1162 to 0.1141 at epoch 2 and drifted up afterwards, while the validation F1 stayed within 0.005 of the frozen model's — the sign that a small learning rate and validation selection are doing their job on a model that has little left to learn from 32 crops of a dataset it trained on. Four epochs (32 steps) take seconds on a T4 and about a minute on a CPU. `TRAINABLE = 'decoders'` also unfreezes the two content decoders (1.75 M parameters).

In [ ]:
EPOCHS = 4  # @param {type:"integer"}
LEARNING_RATE = 1e-5  # @param {type:"number"}
BATCH_SIZE = 4  # @param {type:"integer"}
TRAINABLE = 'change_decoder'  # @param ["change_decoder", "decoders"]

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4), 'val_loss': round(entry['val_loss'], 4)}
    if 'val' in entry:
        row['val_f1'] = entry['val']['f1']
        row['val_iou'] = entry['val']['iou']
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable=TRAINABLE, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'steps': adapt_result['n_steps'], 'best_epoch': adapt_result['best_epoch'], 'loss': adapt_result['loss'], 'precision': adapt_result['precision'], 'batchnorm': adapt_result['batchnorm'], 'seconds': adapt_seconds})

## 7. Held-out evaluation: the paired comparison

The test crops were never used for training or epoch selection — neither here nor by the upstream authors, whose model was trained on the training split and selected on the validation split. The adapted model is scored exactly as the frozen model was in Section 5, and the table puts the baseline, the frozen and the adapted numbers side by side. The cell asserts what the procedure guarantees — the kept epoch's validation loss is no higher than the frozen model's, and re-scoring the validation crops reproduces the kept epoch's F1 within 0.01 (float16 kernels are not bit-reproducible across batch sizes) — and prints the test numbers without asserting a direction: on this sample the changed-class F1 moved from 0.9352 to 0.9352 and the IoU from 0.8782 to 0.8783 in the build record, a sample-sanity observation on 24 crops with no dispersion estimate, not a quality claim. With your own pairs from another city, sensor or season, the gap between frozen and adapted is the number to watch.

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
comparison = {}
for key in ('f1', 'iou', 'precision', 'recall', 'accuracy'):
    comparison[key] = {'baseline_unchanged': frozen_test['baseline_unchanged'][key], 'frozen': frozen_test['model'][key], 'adapted': adapted_test['model'][key]}
for key, row in comparison.items():
    print({key: row})
print({'validation_f1': {'frozen': frozen_val['model']['f1'], 'adapted': adapted_val['model']['f1']}, 'validation_loss': {'frozen': adapt_result['history'][0]['val_loss'], 'kept_epoch': adapt_result['history'][adapt_result['best_epoch']]['val_loss']}})
evaluation_report = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset': dataset_report,
    'frozen': {'test': frozen_test, 'validation': frozen_val},
    'adapted': {'test': adapted_test, 'validation': adapted_val},
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/cfnet_change_detection_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2)
assert adapt_result['history'][adapt_result['best_epoch']]['val_loss'] <= adapt_result['history'][0]['val_loss']
assert abs(adapted_val['model']['f1'] - adapt_result['history'][adapt_result['best_epoch']]['val']['f1']) < 1e-2
print({'report': 'outputs/cfnet_change_detection_evaluation_report.json'})

## 8. Change maps, artifact export and fresh reload

The adapted model's change maps of two held-out pairs are written as PNGs beside the two dates and the reference mask — the binary mask (0 / 255, the BYOD label convention) and the raw `tanh` map rescaled to 0–255 — so they can be opened side by side: the agreement per pair printed here is a sanity check, not an evaluation.

`pipe.save_artifact` writes the trained tensors (about 3.4 MB) as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the converted base file, the adaptation scope, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `CFNetChangePipeline.from_artifact` re-verifies the base file, checks the artifact manifest, scope and digest **before** deserialising, rebuilds the network and overlays the tensors — a fresh object from files, not the in-memory model (VER2). The cell asserts the same held-out F1 within 0.001 and change maps within 0.01 (VER4: float16 tolerances; on one device they are usually identical).

In [ ]:
import platform
import shutil

from PIL import Image

shown_records = test_records[:2]
shown_predictions = pipe.predict(shown_records)
for record, pred in zip(shown_records, shown_predictions['predictions']):
    tag = record['source_id']
    Image.fromarray(record['before']).save(f'outputs/cfnet_change_detection_before_' + tag + '.png')
    Image.fromarray(record['after']).save(f'outputs/cfnet_change_detection_after_' + tag + '.png')
    Image.fromarray(np.where(record['label'] == 1, 255, 0).astype(np.uint8)).save(f'outputs/cfnet_change_detection_label_' + tag + '.png')
    Image.fromarray((pred['mask'] * 255).astype(np.uint8)).save(f'outputs/cfnet_change_detection_change_adapted_' + tag + '.png')
    Image.fromarray(np.clip((pred['change_map'] + 1.0) * 127.5 + 0.5, 0, 255).astype(np.uint8)).save(f'outputs/cfnet_change_detection_map_adapted_' + tag + '.png')
    labelled = record['label'] >= 0
    print({'pair': tag, 'agreement': round(float((pred['mask'] == record['label'])[labelled].mean()), 3), 'change_label': round(float((record['label'] == 1).sum() / labelled.sum()), 3), 'change_predicted': pred['changed_fraction'], 'note': 'sanity check on two pairs'})
with open('outputs/cfnet_change_detection_predictions.json', 'w', encoding='utf-8') as f:
    json.dump({'model': shown_predictions['model'], 'classes': shown_predictions['classes'], 'decision_rule': shown_predictions['decision_rule'], 'predictions': [{'id': p['id'], 'changed_fraction': p['changed_fraction']} for p in shown_predictions['predictions']]}, f, indent=2)

artifact_dir = Path('outputs/cfnet_change_detection_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'cfnet_change_detection', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'trainable': artifact_manifest['adapter']['trainable'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = CFNetChangePipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
reloaded_test = reloaded.evaluate(test_records)
before = pipe.predict(test_records[:2])['predictions']
after = reloaded.predict(test_records[:2])['predictions']
parity = {'f1_diff': round(abs(reloaded_test['model']['f1'] - adapted_test['model']['f1']), 6), 'metrics_identical': reloaded_test['model'] == adapted_test['model'], 'max_abs_map_diff': max(float(np.abs(a['change_map'] - b['change_map']).max()) for a, b in zip(before, after))}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['f1_diff'] < 1e-3 and parity['max_abs_map_diff'] < 1e-2

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model': {**evaluation_report['model'], 'model_license': MODEL_LICENSE, 'device': pipe.device, 'source': pipe.source},
    'provenance': {
        'source_asset': [e for e in MANIFEST['files'] if e['path'] == SOURCE_CKPT_NAME],
        'pickle_audit_sha256': PICKLE_AUDIT_SHA256,
        'converted': verify_converted(WEIGHTS_DIR)['files'],
        'pickle_unpickled_once_for_conversion': True,
        'served_from_pickle': False,
        'remote_code_executed': False,
        'network_source': 'modeling.py carried in this notebook; EfficientNet-B5 stages from torchvision ' + torchvision.__version__ + ' with weights=None',
        'data_tarball': {'name': TAR_NAME, 'sha256': TAR_SHA256, 'pinned_members': 3 * len(SAMPLE_RECORDS)},
        'data_base_url': CORPUS_BASE_URL,
        'data_license': CORPUS_LICENSE,
    },
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__, 'pillow': PIL.__version__, 'numpy': np.__version__},
    'data_source': data_source,
    'comparison': comparison,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes']},
    'reload_parity': parity,
}
with open('outputs/cfnet_change_detection_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

print('outputs/:')
for path in sorted(Path('outputs').rglob('*')):
    if path.is_file():
        print(f'  - {path.as_posix()} ({path.stat().st_size / 1024:.1f} KB)')

## Interpretation and limits

On 24 held-out crops the packaged change detector finds building change with an F1 near 0.93 and an IoU near 0.88, against an all-unchanged baseline that scores 0 on both; a bounded fine-tuning of its change decoder on 32 crops of the training split, selected by validation loss with the frozen model as a candidate, leaves those numbers where they were (0.9352 → 0.9352 F1 in the build record). That is the claim: the adaptation contract runs end to end on real labelled bi-temporal pairs drawn from a digest-verified tarball, the pickle is audited and converted rather than served, the network is carried in plain PyTorch, and the artifact that carries the change is 3.4 MB and reloads with the same outputs. It is not a claim that this sample improves the model — the model already trained on this dataset — nor that 24 crops measure its skill.

The numbers are sample-sanity evidence: one seeded run, 24 crops chosen for having change (the full test split is mostly change-free, which is why the authors' F1 is lower), no dispersion estimate, pixel-pooled metrics that let large buildings dominate, and labels drawn by hand with their own uncertainty at building edges. Nothing here measures the model outside Texas suburbs, outside 0.5 m Google Earth imagery, on pairs that are not co-registered, or on change that is not a building.

Three things to carry to real data. **The two dates are not interchangeable:** the network standardises each date with its own statistics and its content decoders are separate, so `before` and `after` must be the earlier and the later image; a swapped or uncalibrated pair is compared without complaint and silently wrong. **Split by scene, not by crop:** overlapping crops of one scene are near-duplicates, and a random split makes memorisation look like skill. **Read the baseline first:** on a crop with 5 % change the all-unchanged baseline is 95 % accurate; only the changed-class F1, IoU, precision and recall say whether the detector did anything.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify a pickled upstream checkpoint, audit and convert it into safetensors without executing anything outside the audited allow-list, rebuild the network from the carried module, fetch a digest-pinned tarball and extract exactly the pinned labelled pairs, execute bounded fine-tuning, evaluate against a baseline and the frozen model on held-out pairs, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or change-detection skill beyond the checks shown.

**Optional experiments (they do not affect the default path):** set `TRAINABLE = 'decoders'`; raise `EPOCHS` and watch the validation loss drift; try `LEARNING_RATE = 1e-3` to see the validation F1 fall while the frozen model keeps the kept epoch; or bring your own labelled pairs through BYOD and read the baseline before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/cfnet-change-detection-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/cfnet-change-detection-pipeline/blob/main/MODEL_CARD.md
- Weights and conversion notes: https://github.com/kurtvalcorza/cfnet-change-detection-pipeline/blob/main/docs/WEIGHTS.md
- Hugging Face model repository: https://huggingface.co/wifibk/CFNet (revision `c23279428d14186d67ce199b3db358038bf37585`)
- LEVIR-CD dataset (academic use only): https://justchenhao.github.io/LEVIR/ · the CFNet authors' mirror: https://huggingface.co/datasets/wifibk/CFNet_Datasets
- Wu, F., Dong, S., Meng, X. (2025). CFNet: Optimizing remote sensing change detection through content-aware enhancement. arXiv:2503.08505: https://arxiv.org/abs/2503.08505
- Chen, H., Shi, Z. (2020). A spatial-temporal attention-based method and a new dataset for remote sensing image change detection. Remote Sensing 12(10), 1662 (LEVIR-CD)
- Upstream code: https://github.com/wifiBlack/CFNet (the network is vendored in `modeling.py`)
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)